<a href="https://colab.research.google.com/github/amitparikh-108/agentic-ai-ik/blob/main/python/lesson3_live_class_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #0d1117 0%, #1c1f26 50%, #2d1b69 100%); padding: 48px 40px; border-radius: 16px; margin-bottom: 8px;">
  <div style="display:flex; align-items:center; gap:16px; margin-bottom:16px;">
    <span style="font-size:48px;">🚌</span>
    <div>
      <p style="color:#a0aec0; font-size:13px; margin:0; letter-spacing:2px; text-transform:uppercase;">CitySmart · Public Transport Intelligence Platform</p>
      <h1 style="color:#ffffff; font-size:32px; font-weight:700; margin:4px 0 0 0;">Lesson 3 — Real Programs</h1>
    </div>
  </div>
  <p style="color:#b794f4; font-size:16px; margin: 0 0 20px 0;">File I/O · Error Handling · OOP — Classes, Objects, Inheritance</p>
  <div style="display:flex; gap:12px; flex-wrap:wrap;">
    <span style="background:rgba(183,148,244,0.2); color:#b794f4; padding:4px 14px; border-radius:20px; font-size:13px; border:1px solid rgba(183,148,244,0.3);">⏱ 4 hours</span>
    <span style="background:rgba(99,179,237,0.2); color:#90cdf4; padding:4px 14px; border-radius:20px; font-size:13px; border:1px solid rgba(99,179,237,0.3);">🤖 AI-assisted</span>
    <span style="background:rgba(251,211,141,0.2); color:#fbd38d; padding:4px 14px; border-radius:20px; font-size:13px; border:1px solid rgba(251,211,141,0.3);">🏗 Deliverable: OOP System + File Persistence</span>
    <span style="background:rgba(252,129,129,0.2); color:#fc8181; padding:4px 14px; border-radius:20px; font-size:13px; border:1px solid rgba(252,129,129,0.3);">🎯 Beginner → Advanced</span>
  </div>
</div>

## 🗺 What we're building today

Lessons 1 and 2 gave CitySmart a CLI and an analytics engine. But every time you restart Python, everything disappears. Today we fix that — and restructure the entire codebase into proper objects.

By the end of this lesson the system will:
- **Save** the day's trip log to a CSV file and **reload** it next session
- **Never crash** on bad data — every failure is caught and handled gracefully
- Be **refactored into classes**: `Route`, `Trip`, `TransportNetwork` — real OOP
- Use **inheritance**: `ExpressBus` and `NightBus` extend `Route` with custom fare logic

This is the lesson where CitySmart stops being a script and becomes a **real application**.

---

## 📖 How this notebook is structured

Every topic follows the same three-step pattern from Lessons 1 and 2:

| Step | Cell | Looks like | Who does it |
|---|---|---|---|
| 1 — Concept | Markdown | Diagrams + explanations | Instructor explains, learners read |
| 2 — Demo | `🔴 INSTRUCTOR DEMO` code cell | Fully written, runs as-is | **Instructor runs live. Learners watch.** |
| 3 — Exercise | `🟢 YOUR TURN` code cell | `???` gaps to fill | **Learners code independently** |

---

## 🤖 How to use AI in this lesson

> 💬 **AI Prompt boxes** appear before each exercise. Same rules as always:
> 1. Attempt every `???` yourself first — even a wrong guess
> 2. Use AI only after your first attempt
> 3. Read every line of the AI response — never blindly copy-paste
> 4. Close AI and rewrite the solution from memory

This lesson adds a new AI skill: **using AI to generate broken test data** — deliberately malformed CSV files to stress-test your error handling. You will ask AI to be adversarial.

---

<div style="background:#f0fff4; border-left:5px solid #38a169; padding:16px 20px; border-radius:0 10px 10px 0; margin:16px 0;">
  <strong style="color:#276749;">🔗 Setup — Run this cell first, every session</strong><br>
  <span style="color:#2f855a; font-size:14px;">This re-creates the NETWORK and TRIP_LOG from Lessons 1 and 2. We also import the standard library modules used today.</span>
</div>

In [ ]:
# ============================================================
# SETUP — Run this cell before anything else
# ============================================================
import csv        # reading and writing CSV files
import json       # reading and writing JSON files
import os         # file existence checks

NETWORK = {
    "7":  {"name":"Harbour Express", "stops":["Central Bus Stand","Fort","Harbour","Navy Nagar","Colaba"],            "base_fare":20.0,"is_ac":True, "capacity":60,"operator":"BEST"},
    "12": {"name":"Airport Shuttle", "stops":["Majestic","MG Road","Hebbal","Airport T1","Airport T2"],             "base_fare":35.0,"is_ac":True, "capacity":45,"operator":"KSRTC"},
    "31": {"name":"City Loop",       "stops":["KR Market","Shivajinagar","MG Road","Richmond","Jayanagar"],          "base_fare":12.0,"is_ac":False,"capacity":80,"operator":"BMTC"},
    "42": {"name":"North Connector", "stops":["Yeshwanthpur","Peenya","Tumkur Road","Nelamangala","Dobbaspet"],      "base_fare":18.0,"is_ac":False,"capacity":80,"operator":"BMTC"},
    "55": {"name":"South Express",   "stops":["Majestic","Jayanagar","JP Nagar","Bannerghatta Rd","Electronic City"],"base_fare":15.0,"is_ac":True, "capacity":60,"operator":"BMTC"},
    "88": {"name":"Tech Corridor",   "stops":["Silk Board","HSR Layout","Koramangala","Indiranagar","MG Road"],      "base_fare":22.0,"is_ac":True, "capacity":50,"operator":"KSRTC"},
}

TRIP_LOG = []

# Simulate a day of trips (reuse from Lesson 2)
raw_trips = [
    ("7",6,12),("7",7,54),("7",8,60),("7",9,58),("7",12,32),("7",17,55),("7",18,59),("7",21,18),
    ("12",5,8),("12",7,40),("12",8,44),("12",9,43),("12",13,20),("12",17,42),("12",18,45),("12",20,30),
    ("31",6,45),("31",7,78),("31",8,80),("31",9,75),("31",11,40),("31",14,35),("31",17,79),("31",19,72),
    ("42",6,30),("42",7,72),("42",8,78),("42",17,74),("42",18,76),("42",21,28),
    ("55",7,55),("55",8,58),("55",9,52),("55",17,56),("55",18,59),("55",20,35),
    ("88",7,48),("88",8,50),("88",9,49),("88",17,47),("88",18,50),("88",19,44),
]
for rid, hour, pax in raw_trips:
    info    = NETWORK[rid]
    is_peak = (7 <= hour <= 10) or (17 <= hour <= 20)
    fare    = round(pax * info["base_fare"] * (1.15 if is_peak else 1.0), 2)
    TRIP_LOG.append({"route_id":rid,"route_name":info["name"],"hour":hour,
                     "passengers":pax,"capacity":info["capacity"],
                     "fare_collected":fare,"is_peak":is_peak,"is_ac":info["is_ac"]})

print(f"✅ NETWORK: {len(NETWORK)} routes")
print(f"✅ TRIP_LOG: {len(TRIP_LOG)} trips")

---
<div style="background:#faf5ff; border-left:5px solid #805ad5; padding:20px 24px; border-radius:0 12px 12px 0; margin:24px 0 8px 0;">
  <h2 style="color:#553c9a; margin:0 0 4px 0;">Part 1 · File I/O</h2>
  <p style="color:#6b46c1; margin:0; font-size:14px;">⏱ ~45 minutes &nbsp;|&nbsp; Concepts: open(), read, write, with statement, csv module, json module</p>
</div>

### 📁 Why files?

Every variable in Python lives in RAM. When the program ends, RAM is cleared — your `TRIP_LOG` vanishes. Files give your program **memory between sessions**.


### 🔑 The `open()` function and the `with` statement

The `open()` function takes a filename and a mode (`r`, `w`, `a`). Always use the `with` statement — it closes the file automatically, even if an error occurs.


### 📄 CSV files — the standard format for tabular data

CSV = Comma Separated Values. Python's `csv` module handles reading and writing. **Important:** `csv.DictReader` gives you **strings** for every value — you must cast types yourself.


### 🗄 JSON files — for structured / nested data

JSON preserves Python's dict/list structure perfectly. Use `json.dump()` to write and `json.load()` to read.


In [ ]:
#@title 📁 File I/O
from IPython.display import HTML, display
display(HTML("""
<div id="fio_w" style="max-width:620px;margin:16px auto;font-family:'Segoe UI',system-ui,sans-serif;background:linear-gradient(135deg,#1a1a2e 0%,#16213e 50%,#0f3460 100%);border-radius:14px;border:1px solid rgba(99,179,237,0.15);overflow:hidden;user-select:none;">
<div style="padding:14px 20px 6px;display:flex;justify-content:space-between;align-items:center;">
<span style="color:#90cdf4;font-size:13px;letter-spacing:1.5px;text-transform:uppercase;font-weight:600;">📁 File I/O</span>
<span id="fio_w_c" style="color:#4a5568;font-size:12px;font-family:monospace;"></span></div>
<div id="fio_w_s" style="padding:0 10px;min-height:320px;display:flex;align-items:center;justify-content:center;"></div>
<div style="padding:10px 20px 14px;display:flex;justify-content:center;align-items:center;gap:12px;">
<button id="fio_w_p" onclick="fio_w_n(-1)" style="background:rgba(99,179,237,0.1);border:1px solid rgba(99,179,237,0.3);color:#90cdf4;padding:8px 20px;border-radius:8px;cursor:pointer;font-size:14px;font-weight:600;" onmouseover="this.style.background='rgba(99,179,237,0.25)'" onmouseout="this.style.background='rgba(99,179,237,0.1)'">◀ Prev</button>
<div id="fio_w_d" style="display:flex;gap:6px;align-items:center;"></div>
<button id="fio_w_x" onclick="fio_w_n(1)" style="background:rgba(99,179,237,0.2);border:1px solid rgba(99,179,237,0.4);color:#fff;padding:8px 20px;border-radius:8px;cursor:pointer;font-size:14px;font-weight:600;" onmouseover="this.style.background='rgba(99,179,237,0.4)'" onmouseout="this.style.background='rgba(99,179,237,0.2)'">Next ▶</button></div></div>
<script>(function(){const id="fio_w";let step=0;const steps=[`<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 600 220" width="600" height="220" style="font-family:monospace;"><text x="300" y="30" fill="#b794f4" font-size="22" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">Why files?</text><text x="300" y="58" fill="#718096" font-size="13" text-anchor="middle" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">Your data disappears when the program ends</text><rect x="40" y="80" width="240" height="120" rx="10" fill="rgba(252,129,129,0.08)" stroke="rgba(252,129,129,0.3)" stroke-width="2"/><text x="160" y="105" fill="rgb(252,129,129)" font-size="14" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">WITHOUT files</text><text x="160" y="130" fill="rgb(252,129,129)" font-size="12" text-anchor="middle" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">Run → data exists</text><text x="160" y="150" fill="rgb(252,129,129)" font-size="12" text-anchor="middle" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">Close → data GONE</text><text x="160" y="170" fill="#a0aec0" font-size="11" text-anchor="middle" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">Run again → start from zero</text><rect x="320" y="80" width="240" height="120" rx="10" fill="rgba(154,230,180,0.08)" stroke="rgba(154,230,180,0.3)" stroke-width="2"/><text x="440" y="105" fill="rgb(154,230,180)" font-size="14" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">WITH files</text><text x="440" y="130" fill="rgb(154,230,180)" font-size="12" text-anchor="middle" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">Run → save to trips.csv</text><text x="440" y="150" fill="rgb(154,230,180)" font-size="12" text-anchor="middle" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">Close → file safe on disk</text><text x="440" y="170" fill="#a0aec0" font-size="11" text-anchor="middle" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">Run again → load file → data back!</text></svg>`,`<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 600 320" width="600" height="320" style="font-family:monospace;"><text x="300" y="25" fill="#b794f4" font-size="18" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">open(filename, mode)</text><rect x="50" y="60" width="500" height="52" rx="8" fill="rgba(99,179,237,0.06)" stroke="rgba(99,179,237,0.2)" stroke-width="1"/><rect x="65" y="68" width="45" height="35" rx="6" fill="rgba(99,179,237,0.2)" stroke="rgb(99,179,237)" stroke-width="2"/><text x="87" y="88" fill="#fff" font-size="16" text-anchor="middle" font-weight="bold" font-family="monospace" dominant-baseline="middle">"r"</text><text x="130" y="80" fill="rgb(99,179,237)" font-size="15" text-anchor="start" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">Read</text><text x="130" y="100" fill="#a0aec0" font-size="11" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">File must already exist</text><text x="400" y="88" fill="#718096" font-size="11" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">Read trips.csv</text><rect x="50" y="125" width="500" height="52" rx="8" fill="rgba(246,173,85,0.06)" stroke="rgba(246,173,85,0.2)" stroke-width="1"/><rect x="65" y="133" width="45" height="35" rx="6" fill="rgba(246,173,85,0.2)" stroke="rgb(246,173,85)" stroke-width="2"/><text x="87" y="153" fill="#fff" font-size="16" text-anchor="middle" font-weight="bold" font-family="monospace" dominant-baseline="middle">"w"</text><text x="130" y="145" fill="rgb(246,173,85)" font-size="15" text-anchor="start" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">Write</text><text x="130" y="165" fill="#a0aec0" font-size="11" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">Creates new or OVERWRITES</text><text x="400" y="153" fill="#718096" font-size="11" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">Save new report</text><rect x="50" y="190" width="500" height="52" rx="8" fill="rgba(154,230,180,0.06)" stroke="rgba(154,230,180,0.2)" stroke-width="1"/><rect x="65" y="198" width="45" height="35" rx="6" fill="rgba(154,230,180,0.2)" stroke="rgb(154,230,180)" stroke-width="2"/><text x="87" y="218" fill="#fff" font-size="16" text-anchor="middle" font-weight="bold" font-family="monospace" dominant-baseline="middle">"a"</text><text x="130" y="210" fill="rgb(154,230,180)" font-size="15" text-anchor="start" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">Append</text><text x="130" y="230" fill="#a0aec0" font-size="11" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">Adds to end, creates if missing</text><text x="400" y="218" fill="#718096" font-size="11" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">Add today's trips</text><rect x="50" y="265" width="500" height="40" rx="8" fill="rgba(251,211,141,0.08)" stroke="rgba(251,211,141,0.2)" stroke-width="2"/><text x="300" y="288" fill="#fbd38d" font-size="12" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">ALWAYS use 'with' — it closes the file automatically!</text></svg>`,`<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 600 260" width="600" height="260" style="font-family:monospace;"><text x="300" y="25" fill="#63b3ed" font-size="18" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">CSV vs JSON</text><rect x="40" y="50" width="250" height="140" rx="10" fill="rgba(99,179,237,0.08)" stroke="rgba(99,179,237,0.3)" stroke-width="2"/><text x="165" y="72" fill="rgb(99,179,237)" font-size="18" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">CSV</text><text x="60" y="100" fill="#a0aec0" font-size="12" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">Flat tables</text><text x="60" y="120" fill="#a0aec0" font-size="12" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">Each row = same cols</text><text x="60" y="140" fill="#a0aec0" font-size="12" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">Opens in Excel</text><text x="60" y="165" fill="#718096" font-size="11" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">csv.DictWriter / Reader</text><rect x="310" y="50" width="250" height="140" rx="10" fill="rgba(154,230,180,0.08)" stroke="rgba(154,230,180,0.3)" stroke-width="2"/><text x="435" y="72" fill="rgb(154,230,180)" font-size="18" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">JSON</text><text x="330" y="100" fill="#a0aec0" font-size="12" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">Nested structure</text><text x="330" y="120" fill="#a0aec0" font-size="12" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">Preserves types</text><text x="330" y="140" fill="#a0aec0" font-size="12" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">APIs use it</text><text x="330" y="165" fill="#718096" font-size="11" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">json.dump / load</text><rect x="40" y="205" width="520" height="35" rx="6" fill="rgba(251,211,141,0.06)" stroke="rgba(251,211,141,0.2)" stroke-width="2"/><text x="300" y="225" fill="#fbd38d" font-size="12" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">TRIP_LOG → CSV (flat table)    NETWORK → JSON (nested dicts)</text></svg>`];const sg=document.getElementById(id+"_s"),cn=document.getElementById(id+"_c"),dt=document.getElementById(id+"_d"),pb=document.getElementById(id+"_p"),nb=document.getElementById(id+"_x");for(let i=0;i<steps.length;i++){const d=document.createElement("div");d.style.cssText="width:8px;height:8px;border-radius:50%;background:#4a5568;transition:all .2s;cursor:pointer;";d.onclick=function(){step=i;r();};dt.appendChild(d);}function r(){sg.innerHTML=steps[step];cn.textContent="Step "+(step+1)+" of "+steps.length;pb.style.opacity=step===0?"0.3":"1";pb.style.pointerEvents=step===0?"none":"auto";nb.style.opacity=step===steps.length-1?"0.3":"1";nb.style.pointerEvents=step===steps.length-1?"none":"auto";for(let i=0;i<dt.children.length;i++){dt.children[i].style.background=i===step?"#63b3ed":"#4a5568";dt.children[i].style.transform=i===step?"scale(1.4)":"scale(1)";}}window[id+"_n"]=function(dir){step=Math.max(0,Math.min(steps.length-1,step+dir));r();};document.addEventListener("keydown",function(e){const rc=document.getElementById(id).getBoundingClientRect();if(rc.top>window.innerHeight||rc.bottom<0)return;if(e.key==="ArrowRight"||e.key===" "){e.preventDefault();window[id+"_n"](1);}if(e.key==="ArrowLeft"){e.preventDefault();window[id+"_n"](-1);}});r();})();</script>"""))

📁 File I/O 
 
 
 
 ◀ Prev 
 
 Next ▶

---
<div style="background:#1a1a2e; border-left:5px solid #f6ad55; padding:14px 18px; border-radius:0 10px 10px 0; margin:16px 0;">
  <strong style="color:#f6ad55;">🔴 INSTRUCTOR DEMO</strong> &nbsp; <span style="color:#a0aec0; font-size:14px;">Run each cell live while learners watch</span>
</div>


### DEMO — File I/O in CitySmart


In [ ]:
# INSTRUCTOR DEMO — File I/O
# Run this cell and watch the files appear in your working directory

### --- 1. Write TRIP_LOG to a CSV file ---


In [ ]:
FIELDNAMES = ["route_id","route_name","hour","passengers",
              "capacity","fare_collected","is_peak","is_ac"]

with open("trips_demo.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
    writer.writeheader()          # writes the column name row
    writer.writerows(TRIP_LOG)    # writes every trip dict

print("✅ Written trips_demo.csv")
print(f"   File size: {os.path.getsize("trips_demo.csv")} bytes")

### --- 2. Read it back and inspect ---


In [ ]:
loaded_trips = []
with open("trips_demo.csv", "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        # Every value from CSV is a string — must cast back to correct types
        loaded_trips.append({
            "route_id"      : row["route_id"],              # keep as str
            "route_name"    : row["route_name"],
            "hour"          : int(row["hour"]),             # str → int
            "passengers"    : int(row["passengers"]),
            "capacity"      : int(row["capacity"]),
            "fare_collected": float(row["fare_collected"]), # str → float
            "is_peak"       : row["is_peak"] == "True",    # str → bool
            "is_ac"         : row["is_ac"] == "True",
        })

print(f"\n✅ Loaded {len(loaded_trips)} trips from CSV")
print(f"   First row: {loaded_trips[0]}")
print(f"   Types:     hour={type(loaded_trips[0]["hour"])}, "
      f"fare={type(loaded_trips[0]["fare_collected"])}, "
      f"is_peak={type(loaded_trips[0]["is_peak"])}")

### --- 3. Write NETWORK to JSON ---


In [ ]:
with open("network_demo.json", "w") as f:
    json.dump(NETWORK, f, indent=2)

print("\n✅ Written network_demo.json")

### --- 4. Read NETWORK back from JSON ---


In [ ]:
with open("network_demo.json", "r") as f:
    loaded_network = json.load(f)

print(f"✅ Loaded {len(loaded_network)} routes from JSON")
print(f"   Route 7 base_fare type: {type(loaded_network["7"]["base_fare"]).__name__}")
print(f"   ← JSON preserved the float type — no manual casting needed!")

### --- 5. Show raw file content (first 3 lines of CSV) ---


In [ ]:
print("\n--- Raw CSV (first 3 lines) ---")
with open("trips_demo.csv", "r") as f:
    for i, line in enumerate(f):
        print(f"  {line.rstrip()}")
        if i == 2: break

---
<div style="background:#1a3a1a; border-left:5px solid #68d391; padding:14px 18px; border-radius:0 10px 10px 0; margin:16px 0;">
  <strong style="color:#68d391;">🟢 YOUR TURN</strong> &nbsp; <span style="color:#a0aec0; font-size:14px;">Fill in every ??? — attempt before using AI</span>
</div>


### EXERCISE 3.1 — Save and reload CitySmart data (Mandatory)


In [ ]:
# YOUR TURN — Exercise 3.1: Save and reload CitySmart data
# Fill every ??? to complete both functions

FIELDNAMES = ["route_id","route_name","hour","passengers",
              "capacity","fare_collected","is_peak","is_ac"]

### --- TASK 1: Write save_trips() ---


In [ ]:
def save_trips(trips, filename="citysmart_trips.csv"):
    """
    Save a list of trip dicts to a CSV file.

    Parameters:
        trips    : list of trip dicts (e.g. TRIP_LOG)
        filename : str — output file path

    Returns:
        int — number of trips written
    """
    with open(???, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=???)
        writer.???()              # write the header row
        writer.???(trips)         # write all trip dicts at once
    return len(trips)

### --- TASK 2: Write load_trips() ---


In [ ]:
def load_trips(filename="citysmart_trips.csv"):
    """
    Load trips from a CSV file and return as a list of dicts.
    Restores all types correctly — CSV gives strings for everything.

    Returns:
        list of trip dicts with correct Python types
    """
    trips = []
    with open(???, "r") as f:
        reader = csv.???(f)
        for row in reader:
            trips.append({
                "route_id"      : row[???],              # stays str
                "route_name"    : row[???],
                "hour"          : ???(row["hour"]),      # cast to int
                "passengers"    : ???(row["passengers"]),
                "capacity"      : ???(row["capacity"]),
                "fare_collected": ???(row["fare_collected"]),  # cast to float
                "is_peak"       : row["is_peak"] == ???,  # str→bool: compare to "True"
                "is_ac"         : row["is_ac"]   == ???,
            })
    return trips

### --- TASK 3: Round-trip test ---


In [ ]:
# Save TRIP_LOG, load it back, verify nothing changed

n = save_trips(TRIP_LOG)
print(f"✅ Saved {n} trips to citysmart_trips.csv")

reloaded = load_trips()
print(f"✅ Loaded {len(reloaded)} trips from citysmart_trips.csv")

# Spot-check: compare first trip from memory vs loaded from file
original = TRIP_LOG[0]
from_file = reloaded[0]

print("\n=== Round-trip check — first trip ===")
print(f"  route_id      : original={original['route_id']!r}  loaded={from_file['route_id']!r}")
print(f"  passengers    : original={original['passengers']}  loaded={from_file['passengers']}  type={type(from_file['passengers']).__name__}")
print(f"  fare_collected: original={original['fare_collected']}  loaded={from_file['fare_collected']}  type={type(from_file['fare_collected']).__name__}")
print(f"  is_peak       : original={original['is_peak']}  loaded={from_file['is_peak']}  type={type(from_file['is_peak']).__name__}")

assert original["passengers"]     == from_file["passengers"],     "passengers mismatch!"
assert original["fare_collected"] == from_file["fare_collected"], "fare mismatch!"
assert original["is_peak"]        == from_file["is_peak"],        "is_peak mismatch!"
print("\n✅ All values match — round-trip successful!")

---
<div style="background:#fff5f5; border-left:5px solid #e53e3e; padding:20px 24px; border-radius:0 12px 12px 0; margin:24px 0 8px 0;">
  <h2 style="color:#9b2c2c; margin:0 0 4px 0;">Part 2 · Error Handling</h2>
  <p style="color:#c53030; margin:0; font-size:14px;">⏱ ~45 minutes &nbsp;|&nbsp; Concepts: try/except/finally, specific exception types, raising errors, common error types</p>
</div>

### 💥 What is an exception?

When Python cannot execute a line of code, it **raises an exception** — an object that describes what went wrong. If nothing catches it, the program crashes.


### 🛡 try / except / else / finally

The `try` block lets you catch exceptions and handle them gracefully. List specific exceptions **before** broad ones. Catch only what you can actually handle.


### 🚨 Raising your own exceptions

You can `raise` exceptions deliberately to signal bad input. Unlike `print()`, `raise` **forces** the caller to deal with the problem — bugs become visible immediately.


In [ ]:
#@title 💥 Exceptions
from IPython.display import HTML, display
display(HTML("""
<div id="exc_w" style="max-width:620px;margin:16px auto;font-family:'Segoe UI',system-ui,sans-serif;background:linear-gradient(135deg,#1a1a2e 0%,#16213e 50%,#0f3460 100%);border-radius:14px;border:1px solid rgba(99,179,237,0.15);overflow:hidden;user-select:none;">
<div style="padding:14px 20px 6px;display:flex;justify-content:space-between;align-items:center;">
<span style="color:#90cdf4;font-size:13px;letter-spacing:1.5px;text-transform:uppercase;font-weight:600;">💥 Exceptions</span>
<span id="exc_w_c" style="color:#4a5568;font-size:12px;font-family:monospace;"></span></div>
<div id="exc_w_s" style="padding:0 10px;min-height:340px;display:flex;align-items:center;justify-content:center;"></div>
<div style="padding:10px 20px 14px;display:flex;justify-content:center;align-items:center;gap:12px;">
<button id="exc_w_p" onclick="exc_w_n(-1)" style="background:rgba(99,179,237,0.1);border:1px solid rgba(99,179,237,0.3);color:#90cdf4;padding:8px 20px;border-radius:8px;cursor:pointer;font-size:14px;font-weight:600;" onmouseover="this.style.background='rgba(99,179,237,0.25)'" onmouseout="this.style.background='rgba(99,179,237,0.1)'">◀ Prev</button>
<div id="exc_w_d" style="display:flex;gap:6px;align-items:center;"></div>
<button id="exc_w_x" onclick="exc_w_n(1)" style="background:rgba(99,179,237,0.2);border:1px solid rgba(99,179,237,0.4);color:#fff;padding:8px 20px;border-radius:8px;cursor:pointer;font-size:14px;font-weight:600;" onmouseover="this.style.background='rgba(99,179,237,0.4)'" onmouseout="this.style.background='rgba(99,179,237,0.2)'">Next ▶</button></div></div>
<script>(function(){const id="exc_w";let step=0;const steps=[`<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 600 320" width="600" height="320" style="font-family:monospace;"><text x="300" y="25" fill="#fc8181" font-size="20" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">Exception flow</text><text x="140" y="55" fill="rgb(154,230,180)" font-size="14" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">Normal flow</text><rect x="65" y="70" width="150" height="32" rx="4" fill="rgba(154,230,180,0.08)" stroke="rgba(154,230,180,0.2)" stroke-width="2"/><text x="140" y="88" fill="#9ae6b4" font-size="12" text-anchor="middle" font-weight="normal" font-family="monospace" dominant-baseline="middle">line 1</text><rect x="65" y="112" width="150" height="32" rx="4" fill="rgba(154,230,180,0.08)" stroke="rgba(154,230,180,0.2)" stroke-width="2"/><text x="140" y="130" fill="#9ae6b4" font-size="12" text-anchor="middle" font-weight="normal" font-family="monospace" dominant-baseline="middle">line 2</text><rect x="65" y="154" width="150" height="32" rx="4" fill="rgba(154,230,180,0.08)" stroke="rgba(154,230,180,0.2)" stroke-width="2"/><text x="140" y="172" fill="#9ae6b4" font-size="12" text-anchor="middle" font-weight="normal" font-family="monospace" dominant-baseline="middle">line 3</text><rect x="65" y="196" width="150" height="32" rx="4" fill="rgba(154,230,180,0.08)" stroke="rgba(154,230,180,0.2)" stroke-width="2"/><text x="140" y="214" fill="#9ae6b4" font-size="12" text-anchor="middle" font-weight="normal" font-family="monospace" dominant-baseline="middle">line 4</text><text x="140" y="250" fill="#9ae6b4" font-size="11" text-anchor="middle" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">All lines run</text><text x="440" y="55" fill="rgb(252,129,129)" font-size="14" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">When error raised</text><rect x="365" y="70" width="150" height="32" rx="4" fill="rgba(154,230,180,0.06)" stroke="rgba(154,230,180,0.2)" stroke-width="2"/><text x="405" y="88" fill="#9ae6b4" font-size="12" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">line 1</text><text x="500" y="88" fill="#9ae6b4" font-size="10" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">runs</text><rect x="365" y="112" width="150" height="32" rx="4" fill="rgba(252,129,129,0.15)" stroke="rgb(252,129,129)" stroke-width="2"/><text x="405" y="130" fill="#fc8181" font-size="12" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">line 2</text><text x="500" y="130" fill="#fc8181" font-size="10" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">ERROR!</text><rect x="365" y="154" width="150" height="32" rx="4" fill="rgba(74,85,104,0.05)" stroke="#2d3748" stroke-width="2"/><text x="405" y="172" fill="#4a5568" font-size="12" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">line 3</text><text x="500" y="172" fill="#4a5568" font-size="10" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">SKIPPED</text><rect x="365" y="196" width="150" height="32" rx="4" fill="rgba(74,85,104,0.05)" stroke="#2d3748" stroke-width="2"/><text x="405" y="214" fill="#4a5568" font-size="12" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">line 4</text><text x="500" y="214" fill="#4a5568" font-size="10" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">SKIPPED</text><rect x="365" y="245" width="150" height="28" rx="4" fill="rgba(252,129,129,0.1)" stroke="rgba(252,129,129,0.3)" stroke-width="2"/><text x="440" y="261" fill="rgb(252,129,129)" font-size="12" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">CRASH!</text><text x="300" y="295" fill="#fbd38d" font-size="12" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">...unless you CATCH it with try/except</text></svg>`,`<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 600 330" width="600" height="330" style="font-family:monospace;"><text x="300" y="22" fill="#fc8181" font-size="18" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">try / except / else / finally</text><rect x="50" y="50" width="500" height="44" rx="6" fill="rgba(99,179,237,0.06)" stroke="rgba(99,179,237,0.15)" stroke-width="1"/><text x="60" y="68" fill="rgb(99,179,237)" font-size="14" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">try:</text><text x="300" y="68" fill="#718096" font-size="12" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">risky code here</text><text x="300" y="87" fill="rgb(99,179,237)" font-size="10" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">← Runs first</text><rect x="50" y="110" width="500" height="44" rx="6" fill="rgba(252,129,129,0.06)" stroke="rgba(252,129,129,0.15)" stroke-width="1"/><text x="60" y="128" fill="rgb(252,129,129)" font-size="14" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">except ValueError:</text><text x="300" y="128" fill="#718096" font-size="12" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">handle the error</text><text x="300" y="147" fill="rgb(252,129,129)" font-size="10" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">← Only if ValueError</text><rect x="50" y="165" width="500" height="44" rx="6" fill="rgba(246,173,85,0.06)" stroke="rgba(246,173,85,0.15)" stroke-width="1"/><text x="60" y="183" fill="rgb(246,173,85)" font-size="14" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">except FileNotFoundError:</text><text x="300" y="183" fill="#718096" font-size="12" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">handle missing file</text><text x="300" y="202" fill="rgb(246,173,85)" font-size="10" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">← Only if file missing</text><rect x="50" y="220" width="500" height="44" rx="6" fill="rgba(154,230,180,0.06)" stroke="rgba(154,230,180,0.15)" stroke-width="1"/><text x="60" y="238" fill="rgb(154,230,180)" font-size="14" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">else:</text><text x="300" y="238" fill="#718096" font-size="12" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">success code</text><text x="300" y="257" fill="rgb(154,230,180)" font-size="10" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">← Only if NO error</text><rect x="50" y="272" width="500" height="44" rx="6" fill="rgba(251,211,141,0.06)" stroke="rgba(251,211,141,0.15)" stroke-width="1"/><text x="60" y="290" fill="rgb(251,211,141)" font-size="14" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">finally:</text><text x="300" y="290" fill="#718096" font-size="12" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">cleanup code</text><text x="300" y="309" fill="rgb(251,211,141)" font-size="10" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">← ALWAYS runs</text></svg>`,`<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 600 310" width="600" height="310" style="font-family:monospace;"><text x="300" y="22" fill="#76e4f7" font-size="16" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">Exceptions CitySmart encounters</text><rect x="30" y="48" width="540" height="34" rx="4" fill="rgba(252,129,129,0.05)" stroke="rgba(252,129,129,0.15)" stroke-width="1"/><text x="45" y="67" fill="rgb(252,129,129)" font-size="12" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">FileNotFoundError</text><text x="240" y="67" fill="#a0aec0" font-size="11" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">open() on missing file</text><text x="440" y="67" fill="#718096" font-size="10" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">load_trips("x.csv")</text><rect x="30" y="90" width="540" height="34" rx="4" fill="rgba(246,173,85,0.05)" stroke="rgba(246,173,85,0.15)" stroke-width="1"/><text x="45" y="109" fill="rgb(246,173,85)" font-size="12" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">ValueError</text><text x="240" y="109" fill="#a0aec0" font-size="11" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">bad type conversion</text><text x="440" y="109" fill="#718096" font-size="10" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">int("forty-three")</text><rect x="30" y="132" width="540" height="34" rx="4" fill="rgba(183,148,244,0.05)" stroke="rgba(183,148,244,0.15)" stroke-width="1"/><text x="45" y="151" fill="rgb(183,148,244)" font-size="12" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">KeyError</text><text x="240" y="151" fill="#a0aec0" font-size="11" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">missing dict key</text><text x="440" y="151" fill="#718096" font-size="10" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">NETWORK["99"]</text><rect x="30" y="174" width="540" height="34" rx="4" fill="rgba(251,211,141,0.05)" stroke="rgba(251,211,141,0.15)" stroke-width="1"/><text x="45" y="193" fill="rgb(251,211,141)" font-size="12" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">TypeError</text><text x="240" y="193" fill="#a0aec0" font-size="11" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">wrong type for operation</text><text x="440" y="193" fill="#718096" font-size="10" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">"abc" + 42</text><rect x="30" y="216" width="540" height="34" rx="4" fill="rgba(252,129,129,0.05)" stroke="rgba(252,129,129,0.15)" stroke-width="1"/><text x="45" y="235" fill="rgb(252,129,129)" font-size="12" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">ZeroDivisionError</text><text x="240" y="235" fill="#a0aec0" font-size="11" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">dividing by zero</text><text x="440" y="235" fill="#718096" font-size="10" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">revenue / 0</text><rect x="30" y="258" width="540" height="34" rx="4" fill="rgba(99,179,237,0.05)" stroke="rgba(99,179,237,0.15)" stroke-width="1"/><text x="45" y="277" fill="rgb(99,179,237)" font-size="12" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">IndexError</text><text x="240" y="277" fill="#a0aec0" font-size="11" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">list index out of range</text><text x="440" y="277" fill="#718096" font-size="10" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">stops[99]</text></svg>`,`<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 600 290" width="600" height="290" style="font-family:monospace;"><text x="300" y="22" fill="#f6ad55" font-size="16" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">Raising your own exceptions</text><rect x="40" y="48" width="520" height="55" rx="8" fill="rgba(246,173,85,0.06)" stroke="rgba(246,173,85,0.2)" stroke-width="2"/><text x="60" y="70" fill="#e2e8f0" font-size="14" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">raise ValueError(</text><text x="270" y="70" fill="rgb(246,173,85)" font-size="12" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">"passengers cannot be negative"</text><text x="530" y="70" fill="#e2e8f0" font-size="14" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">)</text><text x="60" y="90" fill="#a0aec0" font-size="11" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">Forces caller to handle it — unlike print()!</text><text x="300" y="130" fill="#b794f4" font-size="15" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">Custom exception classes</text><rect x="60" y="150" width="480" height="34" rx="4" fill="rgba(183,148,244,0.06)" stroke="rgba(183,148,244,0.15)" stroke-width="1"/><text x="80" y="169" fill="rgb(183,148,244)" font-size="12" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">class CitSmartError(Exception)</text><text x="360" y="169" fill="#718096" font-size="11" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">Base class for all errors</text><rect x="60" y="192" width="480" height="34" rx="4" fill="rgba(99,179,237,0.06)" stroke="rgba(99,179,237,0.15)" stroke-width="1"/><text x="80" y="211" fill="rgb(99,179,237)" font-size="12" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">class RouteNotFoundError</text><text x="360" y="211" fill="#718096" font-size="11" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">route_id not in NETWORK</text><rect x="60" y="234" width="480" height="34" rx="4" fill="rgba(252,129,129,0.06)" stroke="rgba(252,129,129,0.15)" stroke-width="1"/><text x="80" y="253" fill="rgb(252,129,129)" font-size="12" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">class CapacityExceededError</text><text x="360" y="253" fill="#718096" font-size="11" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">passengers &gt; capacity</text></svg>`];const sg=document.getElementById(id+"_s"),cn=document.getElementById(id+"_c"),dt=document.getElementById(id+"_d"),pb=document.getElementById(id+"_p"),nb=document.getElementById(id+"_x");for(let i=0;i<steps.length;i++){const d=document.createElement("div");d.style.cssText="width:8px;height:8px;border-radius:50%;background:#4a5568;transition:all .2s;cursor:pointer;";d.onclick=function(){step=i;r();};dt.appendChild(d);}function r(){sg.innerHTML=steps[step];cn.textContent="Step "+(step+1)+" of "+steps.length;pb.style.opacity=step===0?"0.3":"1";pb.style.pointerEvents=step===0?"none":"auto";nb.style.opacity=step===steps.length-1?"0.3":"1";nb.style.pointerEvents=step===steps.length-1?"none":"auto";for(let i=0;i<dt.children.length;i++){dt.children[i].style.background=i===step?"#63b3ed":"#4a5568";dt.children[i].style.transform=i===step?"scale(1.4)":"scale(1)";}}window[id+"_n"]=function(dir){step=Math.max(0,Math.min(steps.length-1,step+dir));r();};document.addEventListener("keydown",function(e){const rc=document.getElementById(id).getBoundingClientRect();if(rc.top>window.innerHeight||rc.bottom<0)return;if(e.key==="ArrowRight"||e.key===" "){e.preventDefault();window[id+"_n"](1);}if(e.key==="ArrowLeft"){e.preventDefault();window[id+"_n"](-1);}});r();})();</script>"""))

💥 Exceptions 
 
 
 
 ◀ Prev 
 
 Next ▶

---
<div style="background:#1a1a2e; border-left:5px solid #f6ad55; padding:14px 18px; border-radius:0 10px 10px 0; margin:16px 0;">
  <strong style="color:#f6ad55;">🔴 INSTRUCTOR DEMO</strong> &nbsp; <span style="color:#a0aec0; font-size:14px;">Run each cell live while learners watch</span>
</div>


### DEMO — Error handling in CitySmart


In [ ]:
# INSTRUCTOR DEMO — Error handling
# Watch how each exception type is caught differently

### --- 1. FileNotFoundError ---


In [ ]:
print("=== 1. FileNotFoundError ===")
try:
    with open("no_such_file.csv", "r") as f:
        data = f.read()
except FileNotFoundError as e:
    print(f"  Caught FileNotFoundError: {e}")
    print(f"  Filename was: {e.filename}")

### --- 2. ValueError during type casting ---


In [ ]:
print("\n=== 2. ValueError in CSV casting ===")
bad_row = {"route_id":"7", "passengers":"forty", "fare_collected":"18.50"}
try:
    passengers = int(bad_row["passengers"])    # "forty" → crash!
except ValueError as e:
    print(f"  Caught ValueError: {e}")
    print(f"  Bad value was: {bad_row['passengers']!r}")

### --- 3. KeyError on missing route ---


In [ ]:
print("\n=== 3. KeyError on missing route ===")
try:
    info = NETWORK["99"]    # route 99 doesn't exist
except KeyError as e:
    print(f"  Caught KeyError: {e}")

### --- 4. Multiple exceptions — resilient CSV loader ---


In [ ]:
print("\n=== 4. Resilient CSV loader ===")

def load_trips_safe(filename):
    """Load CSV and skip any rows that cannot be parsed."""
    trips  = []
    errors = []
    try:
        with open(filename, "r") as f:
            reader = csv.DictReader(f)
            for i, row in enumerate(reader, start=2):  # row 1 = header
                try:
                    trips.append({
                        "route_id"      : row["route_id"],
                        "passengers"    : int(row["passengers"]),
                        "fare_collected": float(row["fare_collected"]),
                        "is_peak"       : row["is_peak"] == "True",
                    })
                except (ValueError, KeyError) as e:
                    errors.append(f"  Row {i}: {e}")
    except FileNotFoundError:
        print(f"  File not found: {filename}")
        return [], []

    if errors:
        print(f"  ⚠ Skipped {len(errors)} bad rows:")
        for err in errors:
            print(err)
    return trips, errors

trips, errs = load_trips_safe("citysmart_trips.csv")  # our clean file
print(f"  Loaded {len(trips)} trips, {len(errs)} errors")

### --- 5. Raising custom exceptions ---


In [ ]:
print("\n=== 5. Raising custom exceptions ===")

class RouteNotFoundError(Exception):
    pass

class CapacityExceededError(Exception):
    pass

def log_trip_strict(route_id, hour, passengers):
    """Strict version — raises exceptions on bad input."""
    if route_id not in NETWORK:
        raise RouteNotFoundError(f"Route '{route_id}' not in NETWORK")
    cap = NETWORK[route_id]["capacity"]
    if passengers > cap:
        raise CapacityExceededError(
            f"Route {route_id}: {passengers} passengers exceeds capacity {cap}")
    if passengers < 0:
        raise ValueError(f"passengers cannot be negative: {passengers}")
    print(f"  ✅ Trip logged: Route {route_id}, {passengers} passengers")

for test in [("7", 8, 45), ("99", 8, 20), ("31", 9, 95), ("12", 7, -5)]:
    try:
        log_trip_strict(*test)
    except RouteNotFoundError as e:
        print(f"  ❌ RouteNotFoundError: {e}")
    except CapacityExceededError as e:
        print(f"  ❌ CapacityExceededError: {e}")
    except ValueError as e:
        print(f"  ❌ ValueError: {e}")

---
<div style="background:#1a3a1a; border-left:5px solid #68d391; padding:14px 18px; border-radius:0 10px 10px 0; margin:16px 0;">
  <strong style="color:#68d391;">🟢 YOUR TURN</strong> &nbsp; <span style="color:#a0aec0; font-size:14px;">Fill in every ??? — attempt before using AI</span>
</div>


### EXERCISE 3.2 — Resilient CSV loader (Mandatory)


In [ ]:
# YOUR TURN — Exercise 3.2: Resilient CSV loader
# Fill every ??? to build a loader that never crashes

### --- TASK 1: Define your custom exceptions ---


In [ ]:
class CitySrnartError(Exception):
    """Base class for all CitySmart errors."""
    pass

class RouteNotFoundError(CitySrnartError):
    """Raised when a route_id does not exist in NETWORK."""
    pass

class CapacityExceededError(CitySrnartError):
    """Raised when passengers exceed bus capacity."""
    pass

### --- TASK 2: Paste the AI-generated bad CSV here ---


In [ ]:
# Replace the content below with what AI gave you
BAD_CSV_CONTENT = """route_id,route_name,hour,passengers,capacity,fare_collected,is_peak,is_ac
7,Harbour Express,8,60,60,1380.0,True,True
12,Airport Shuttle,7,forty,45,1610.0,True,True
31,City Loop,9,80,80,1104.0,True,False
42,North Connector,8,95,80,1620.0,True,False
55,South Express,,58,60,870.0,True,True
88,Tech Corridor,17,-5,50,940.0,Truee,True
7,Harbour Express,21,18,60,360.0,False,True
"""

# Write it to a file so we can test our loader
with open("bad_trips.csv", "w") as f:
    f.write(BAD_CSV_CONTENT)
print("✅ Wrote bad_trips.csv")

### --- TASK 3: Build the resilient loader ---


In [ ]:
def load_trips_resilient(filename):
    """
    Load a CSV of trips, skipping bad rows gracefully.

    Handles:
    - FileNotFoundError (file missing)
    - ValueError (bad number conversion)
    - KeyError (missing CSV column)
    - Validation: passengers must be 0–capacity, hour must be 0–23

    Returns:
        (trips: list, errors: list of str)
    """
    trips  = []
    errors = []

    try:
        with open(???, "r") as f:
            reader = csv.DictReader(f)
            for i, row in enumerate(reader, start=2):
                try:
                    # Cast all values — each cast can raise ValueError or KeyError
                    route_id  = row[???]
                    hour      = ???(row["hour"])       # may raise ValueError
                    passengers= ???(row["passengers"]) # may raise ValueError
                    capacity  = ???(row["capacity"])
                    fare      = ???(row["fare_collected"])
                    is_peak   = row["is_peak"] == "True"
                    is_ac     = row["is_ac"]   == "True"

                    # Validate values
                    if not (0 <= hour <= 23):
                        raise ValueError(f"hour {hour} out of range 0–23")
                    if passengers < ???:
                        raise ValueError(f"passengers cannot be negative: {passengers}")
                    if passengers > ???:
                        raise CapacityExceededError(
                            f"passengers {passengers} > capacity {capacity}")
                    if fare < ???:
                        raise ValueError(f"fare cannot be negative: {fare}")

                    trips.append({
                        "route_id": route_id, "hour": hour,
                        "passengers": passengers, "capacity": capacity,
                        "fare_collected": fare, "is_peak": is_peak, "is_ac": is_ac,
                    })

                except ??? as e:   # catch ValueError AND KeyError together
                    errors.append(f"Row {i} skipped — ValueError/KeyError: {e}")
                except CapacityExceededError as e:
                    errors.append(f"Row {i} skipped — CapacityExceededError: {e}")

    except ???:
        print(f"❌ File not found: {filename}")
        return [], []

    return trips, errors

SyntaxError: invalid syntax (1400097503.py, line 18)

### --- TASK 4: Test the loader ---


In [ ]:
good, errs = load_trips_resilient("bad_trips.csv")

print(f"\n=== Loader results ===")
print(f"  ✅ Accepted : {len(good)} valid trips")
print(f"  ⚠ Rejected : {len(errs)} bad rows")
print("\n  Error log:")
for e in errs:
    print(f"    {e}")

print("\n  Valid trips loaded:")
for t in good:
    print(f"    Route {t['route_id']} at {t['hour']:02d}:00 — {t['passengers']} pax, ₹{t['fare_collected']}")

---
<div style="background:#ebf8ff; border-left:5px solid #3182ce; padding:20px 24px; border-radius:0 12px 12px 0; margin:24px 0 8px 0;">
  <h2 style="color:#2b6cb0; margin:0 0 4px 0;">Part 3 · OOP — Classes & Objects</h2>
  <p style="color:#2c5282; margin:0; font-size:14px;">⏱ ~45 minutes &nbsp;|&nbsp; Concepts: class, __init__, self, instance methods, __str__, __repr__</p>
</div>

### 🏗 What is a class?

A class is a **blueprint** for creating objects. An object bundles together **data** (attributes) and **behaviour** (methods) that belong together.


### 🔬 Anatomy of a class

Every class has a constructor (`__init__`), instance attributes (`self.name`), instance methods, and display methods (`__str__`, `__repr__`).


In [ ]:
#@title 🏗 Class anatomy
from IPython.display import HTML, display
display(HTML("""
<div id="cls_w" style="max-width:620px;margin:16px auto;font-family:'Segoe UI',system-ui,sans-serif;background:linear-gradient(135deg,#1a1a2e 0%,#16213e 50%,#0f3460 100%);border-radius:14px;border:1px solid rgba(99,179,237,0.15);overflow:hidden;user-select:none;">
<div style="padding:14px 20px 6px;display:flex;justify-content:space-between;align-items:center;">
<span style="color:#90cdf4;font-size:13px;letter-spacing:1.5px;text-transform:uppercase;font-weight:600;">🏗 Class anatomy</span>
<span id="cls_w_c" style="color:#4a5568;font-size:12px;font-family:monospace;"></span></div>
<div id="cls_w_s" style="padding:0 10px;min-height:340px;display:flex;align-items:center;justify-content:center;"></div>
<div style="padding:10px 20px 14px;display:flex;justify-content:center;align-items:center;gap:12px;">
<button id="cls_w_p" onclick="cls_w_n(-1)" style="background:rgba(99,179,237,0.1);border:1px solid rgba(99,179,237,0.3);color:#90cdf4;padding:8px 20px;border-radius:8px;cursor:pointer;font-size:14px;font-weight:600;" onmouseover="this.style.background='rgba(99,179,237,0.25)'" onmouseout="this.style.background='rgba(99,179,237,0.1)'">◀ Prev</button>
<div id="cls_w_d" style="display:flex;gap:6px;align-items:center;"></div>
<button id="cls_w_x" onclick="cls_w_n(1)" style="background:rgba(99,179,237,0.2);border:1px solid rgba(99,179,237,0.4);color:#fff;padding:8px 20px;border-radius:8px;cursor:pointer;font-size:14px;font-weight:600;" onmouseover="this.style.background='rgba(99,179,237,0.4)'" onmouseout="this.style.background='rgba(99,179,237,0.2)'">Next ▶</button></div></div>
<script>(function(){const id="cls_w";let step=0;const steps=[`<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 600 265" width="600" height="265" style="font-family:monospace;"><text x="300" y="25" fill="#63b3ed" font-size="20" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">Why classes?</text><rect x="30" y="50" width="250" height="160" rx="10" fill="rgba(252,129,129,0.06)" stroke="rgba(252,129,129,0.2)" stroke-width="2"/><text x="155" y="72" fill="rgb(252,129,129)" font-size="13" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">WITHOUT OOP</text><text x="45" y="95" fill="#a0aec0" font-size="10" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">route_7_name = ...</text><text x="45" y="115" fill="#a0aec0" font-size="10" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">route_7_fare = ...</text><text x="45" y="135" fill="#a0aec0" font-size="10" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">route_7_stops = ...</text><text x="45" y="155" fill="#a0aec0" font-size="10" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">route_12_name = ...</text><text x="45" y="175" fill="#a0aec0" font-size="10" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">route_12_fare = ...</text><text x="45" y="195" fill="#a0aec0" font-size="10" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">... (copy for each!)</text><rect x="320" y="50" width="250" height="160" rx="10" fill="rgba(154,230,180,0.06)" stroke="rgba(154,230,180,0.2)" stroke-width="2"/><text x="445" y="72" fill="rgb(154,230,180)" font-size="13" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">WITH OOP</text><text x="335" y="95" fill="rgb(154,230,180)" font-size="10" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">class Route:</text><text x="335" y="115" fill="rgb(154,230,180)" font-size="10" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">  def __init__(self,...):</text><text x="335" y="135" fill="rgb(154,230,180)" font-size="10" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">    self.name = name</text><text x="335" y="155" fill="rgb(154,230,180)" font-size="10" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle"></text><text x="335" y="175" fill="rgb(154,230,180)" font-size="10" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">r7  = Route("7",...)</text><text x="335" y="195" fill="rgb(154,230,180)" font-size="10" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">r12 = Route("12",...)</text><text x="300" y="235" fill="#fbd38d" font-size="12" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">All route logic in one reusable blueprint</text></svg>`,`<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 600 140" width="600" height="140" style="font-family:monospace;"><text x="300" y="20" fill="#63b3ed" font-size="16" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">Class anatomy — step by step</text><rect x="40" y="42" width="520" height="30" rx="6" fill="rgba(99,179,237,0.15)" stroke="rgb(99,179,237)" stroke-width="2"/><text x="60" y="59" fill="#fff" font-size="16" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">class Route:</text><text x="250" y="59" fill="#90cdf4" font-size="12" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">← 'class' keyword + name</text><text x="300" y="100" fill="#a0aec0" font-size="13" text-anchor="middle" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">The blueprint — no object exists yet</text></svg>`,`<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 600 230" width="600" height="230" style="font-family:monospace;"><text x="300" y="20" fill="#63b3ed" font-size="16" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">Class anatomy — __init__</text><rect x="40" y="42" width="520" height="30" rx="6" fill="rgba(74,85,104,0.1)" stroke="#2d3748" stroke-width="1"/><text x="60" y="59" fill="#718096" font-size="14" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">class Route:</text><rect x="40" y="82" width="520" height="65" rx="6" fill="rgba(154,230,180,0.1)" stroke="rgb(154,230,180)" stroke-width="2"/><text x="60" y="100" fill="#fff" font-size="13" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">def __init__(self, route_id, name, fare):</text><text x="60" y="128" fill="#9ae6b4" font-size="13" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">    self.name = name</text><text x="400" y="100" fill="rgb(154,230,180)" font-size="11" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">← constructor</text><text x="400" y="128" fill="rgb(154,230,180)" font-size="11" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">← instance attribute</text><rect x="40" y="165" width="520" height="45" rx="6" fill="rgba(251,211,141,0.06)" stroke="rgba(251,211,141,0.2)" stroke-width="2"/><text x="60" y="182" fill="#fbd38d" font-size="14" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">self</text><text x="100" y="182" fill="#a0aec0" font-size="13" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">= the specific object being created</text><text x="60" y="200" fill="#718096" font-size="10" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">Python passes it automatically — you never type it in the call</text></svg>`,`<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 600 190" width="600" height="190" style="font-family:monospace;"><text x="300" y="20" fill="#63b3ed" font-size="16" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">Class anatomy — methods</text><rect x="40" y="42" width="520" height="30" rx="6" fill="rgba(74,85,104,0.1)" stroke="#2d3748" stroke-width="1"/><text x="60" y="59" fill="#718096" font-size="14" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">class Route:</text><rect x="40" y="78" width="520" height="25" rx="4" fill="rgba(74,85,104,0.05)" stroke="#2d3748" stroke-width="1"/><text x="60" y="92" fill="#4a5568" font-size="12" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">def __init__(self, ...): ...</text><rect x="40" y="112" width="520" height="55" rx="6" fill="rgba(183,148,244,0.1)" stroke="rgb(183,148,244)" stroke-width="2"/><text x="60" y="130" fill="#fff" font-size="13" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">def get_fare(self, stops, is_peak=False):</text><text x="60" y="155" fill="#b794f4" font-size="12" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">    return self.base_fare + extra</text><text x="400" y="130" fill="#b794f4" font-size="11" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">← instance method</text><text x="400" y="155" fill="#b794f4" font-size="11" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">← uses self's data</text></svg>`,`<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 600 195" width="600" height="195" style="font-family:monospace;"><text x="300" y="20" fill="#63b3ed" font-size="16" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">Class anatomy — __str__ and __repr__</text><rect x="40" y="48" width="520" height="55" rx="8" fill="rgba(118,228,247,0.08)" stroke="rgba(118,228,247,0.3)" stroke-width="2"/><text x="60" y="68" fill="#76e4f7" font-size="15" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">__str__</text><text x="140" y="68" fill="#a0aec0" font-size="12" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">→ human-friendly, called by print()</text><text x="60" y="90" fill="#76e4f7" font-size="12" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">"Route 7 — Harbour Express [AC]"</text><rect x="40" y="115" width="520" height="55" rx="8" fill="rgba(246,173,85,0.08)" stroke="rgba(246,173,85,0.3)" stroke-width="2"/><text x="60" y="135" fill="#f6ad55" font-size="15" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">__repr__</text><text x="150" y="135" fill="#a0aec0" font-size="12" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">→ developer-friendly, for debugging</text><text x="60" y="157" fill="#f6ad55" font-size="12" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">"Route(id=7, name=Harbour Express, fare=20.0)"</text></svg>`,`<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 600 245" width="600" height="245" style="font-family:monospace;"><text x="300" y="20" fill="#9ae6b4" font-size="16" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">Creating and using objects</text><text x="300" y="45" fill="#718096" font-size="12" text-anchor="middle" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">Creating:</text><rect x="40" y="55" width="520" height="35" rx="6" fill="rgba(154,230,180,0.08)" stroke="rgba(154,230,180,0.2)" stroke-width="2"/><text x="60" y="75" fill="#e2e8f0" font-size="12" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">r7 = Route("7", "Harbour Express", 20.0, [...], True)</text><text x="300" y="110" fill="#718096" font-size="12" text-anchor="middle" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">Using:</text><rect x="40" y="120" width="520" height="30" rx="4" fill="rgba(99,179,237,0.05)" stroke="rgba(99,179,237,0.15)" stroke-width="1"/><text x="55" y="137" fill="#e2e8f0" font-size="13" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">r7.name</text><text x="310" y="137" fill="rgb(99,179,237)" font-size="12" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">→ "Harbour Express"</text><text x="480" y="137" fill="#718096" font-size="9" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">attribute access</text><rect x="40" y="158" width="520" height="30" rx="4" fill="rgba(154,230,180,0.05)" stroke="rgba(154,230,180,0.15)" stroke-width="1"/><text x="55" y="175" fill="#e2e8f0" font-size="13" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">r7.get_fare(8, True)</text><text x="310" y="175" fill="rgb(154,230,180)" font-size="12" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">→ 27.625</text><text x="480" y="175" fill="#718096" font-size="9" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">method call</text><rect x="40" y="196" width="520" height="30" rx="4" fill="rgba(183,148,244,0.05)" stroke="rgba(183,148,244,0.15)" stroke-width="1"/><text x="55" y="213" fill="#e2e8f0" font-size="13" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">print(r7)</text><text x="310" y="213" fill="rgb(183,148,244)" font-size="12" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">→ "Route 7 — Harbour..."</text><text x="480" y="213" fill="#718096" font-size="9" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">calls __str__</text></svg>`];const sg=document.getElementById(id+"_s"),cn=document.getElementById(id+"_c"),dt=document.getElementById(id+"_d"),pb=document.getElementById(id+"_p"),nb=document.getElementById(id+"_x");for(let i=0;i<steps.length;i++){const d=document.createElement("div");d.style.cssText="width:8px;height:8px;border-radius:50%;background:#4a5568;transition:all .2s;cursor:pointer;";d.onclick=function(){step=i;r();};dt.appendChild(d);}function r(){sg.innerHTML=steps[step];cn.textContent="Step "+(step+1)+" of "+steps.length;pb.style.opacity=step===0?"0.3":"1";pb.style.pointerEvents=step===0?"none":"auto";nb.style.opacity=step===steps.length-1?"0.3":"1";nb.style.pointerEvents=step===steps.length-1?"none":"auto";for(let i=0;i<dt.children.length;i++){dt.children[i].style.background=i===step?"#63b3ed":"#4a5568";dt.children[i].style.transform=i===step?"scale(1.4)":"scale(1)";}}window[id+"_n"]=function(dir){step=Math.max(0,Math.min(steps.length-1,step+dir));r();};document.addEventListener("keydown",function(e){const rc=document.getElementById(id).getBoundingClientRect();if(rc.top>window.innerHeight||rc.bottom<0)return;if(e.key==="ArrowRight"||e.key===" "){e.preventDefault();window[id+"_n"](1);}if(e.key==="ArrowLeft"){e.preventDefault();window[id+"_n"](-1);}});r();})();</script>"""))

🏗 Class anatomy 
 
 
 
 ◀ Prev 
 
 Next ▶

---
<div style="background:#1a1a2e; border-left:5px solid #f6ad55; padding:14px 18px; border-radius:0 10px 10px 0; margin:16px 0;">
  <strong style="color:#f6ad55;">🔴 INSTRUCTOR DEMO</strong> &nbsp; <span style="color:#a0aec0; font-size:14px;">Run each cell live while learners watch</span>
</div>


### DEMO — The Route class in CitySmart


In [ ]:
# INSTRUCTOR DEMO — The Route class
# Watch how a plain dict transforms into a proper object

class Route:
    """Represents a single CitySmart bus route."""

    def __init__(self, route_id, name, base_fare, stops, is_ac, capacity, operator):
        # Validate on creation — fail fast rather than silently store bad data
        if base_fare <= 0:
            raise ValueError(f"base_fare must be > 0, got {base_fare}")
        if not stops:
            raise ValueError("stops list cannot be empty")

        # Store every value as an instance attribute
        self.route_id = route_id
        self.name     = name
        self.base_fare= base_fare
        self.stops    = stops
        self.is_ac    = is_ac
        self.capacity = capacity
        self.operator = operator

    # ── Instance methods ────────────────────────────────────
    def get_fare(self, stops_travelled, is_peak=False):
        """Calculate fare for a given number of stops."""
        extra = max(0, stops_travelled - 5) * 2.50
        total = self.base_fare + extra
        if is_peak:
            total *= 1.15
        if self.is_ac:
            total += 8.0
        return round(total, 2)

    def get_stop_index(self, stop_name):
        """Return index of a stop, or -1 if not found (case-insensitive)."""
        for i, stop in enumerate(self.stops):
            if stop.lower() == stop_name.lower():
                return i
        return -1

    def serves_stop(self, stop_name):
        """Return True if this route serves the given stop name."""
        return self.get_stop_index(stop_name) >= 0

    # ── Display methods ─────────────────────────────────────
    def __str__(self):
        ac = "AC" if self.is_ac else "Non-AC"
        return f"Route {self.route_id} — {self.name} [{ac}, {self.operator}] ₹{self.base_fare}"

    def __repr__(self):
        return f"Route(id={self.route_id!r}, name={self.name!r}, fare={self.base_fare})"

### --- Create Route objects from NETWORK dict ---


In [ ]:
routes_as_objects = {}
for rid, info in NETWORK.items():
    routes_as_objects[rid] = Route(
        route_id = rid,
        name     = info["name"],
        base_fare= info["base_fare"],
        stops    = info["stops"],
        is_ac    = info["is_ac"],
        capacity = info["capacity"],
        operator = info["operator"],
    )

### --- Use the objects ---


In [ ]:
r7  = routes_as_objects["7"]
r12 = routes_as_objects["12"]

print("=== Object display ===")
print(r7)          # calls __str__
print(repr(r7))    # calls __repr__

print("\n=== Attribute access ===")
print(f"Name        : {r7.name}")
print(f"Stops       : {r7.stops}")
print(f"First stop  : {r7.stops[0]}")
print(f"Last stop   : {r7.stops[-1]}")

print("\n=== Method calls ===")
print(f"Fare, 6 stops, off-peak : ₹{r12.get_fare(6)}")
print(f"Fare, 6 stops, peak     : ₹{r12.get_fare(6, is_peak=True)}")
print(f"Serves MG Road?         : {r12.serves_stop('MG Road')}")
print(f"Serves Hebbal?          : {r12.serves_stop('hebbal')}")
print(f"Serves Dobbaspet?       : {r12.serves_stop('Dobbaspet')}")

print("\n=== All routes as objects ===")
for rid, route in routes_as_objects.items():
    print(f"  {route}")

---
<div style="background:#1a3a1a; border-left:5px solid #68d391; padding:14px 18px; border-radius:0 10px 10px 0; margin:16px 0;">
  <strong style="color:#68d391;">🟢 YOUR TURN</strong> &nbsp; <span style="color:#a0aec0; font-size:14px;">Fill in every ??? — attempt before using AI</span>
</div>


### EXERCISE 3.3 — The Trip class (Mandatory)


In [ ]:
# YOUR TURN — Exercise 3.3: The Trip class
# Fill every ??? to build a complete Trip object

class Trip:
    """
    Represents one CitySmart bus journey.

    Attributes (all set in __init__):
        route        : Route object
        hour         : int  0–23
        passengers   : int  0 to route.capacity
        is_peak      : bool (computed from hour)
        fare_collected: float (computed from route.get_fare)
    """

    # ── Constructor ─────────────────────────────────────────
    def __init__(self, route, hour, passengers):
        """
        Parameters:
            route      : Route object (already created)
            hour       : int 0–23
            passengers : int 0 to route.capacity
        Raises:
            TypeError  if route is not a Route instance
            ValueError if hour or passengers are out of range
        """
        # Validate route is a Route object
        if not isinstance(route, Route):
            raise TypeError(f"route must be a Route object, got {type(route).__name__}")

        # Validate hour
        if not (??? <= hour <= ???):
            raise ValueError(f"hour must be 0–23, got {hour}")

        # Validate passengers
        if passengers < ???:
            raise ValueError(f"passengers cannot be negative: {passengers}")
        if passengers > ???.capacity:
            raise ValueError(
                f"passengers {passengers} exceeds capacity {route.capacity}")

        # Store attributes
        self.route      = ???
        self.hour       = ???
        self.passengers = ???

        # Compute derived attributes
        self.is_peak = (7 <= hour <= 10) or (17 <= hour <= 20)
        stops_on_route   = len(???.stops) - 1   # stops travelled = all stops
        self.fare_collected = ???.get_fare(stops_on_route, self.is_peak)

    # ── Computed properties ──────────────────────────────────
    def load_pct(self):
        """Return bus load as a percentage (0.0–100.0+)."""
        return round((self.??? / self.route.capacity) * 100, 1)

    def is_overcrowded(self):
        """Return True if load exceeds 95%."""
        return self.load_pct() >= ???

    def revenue_per_passenger(self):
        """Return fare_collected divided by passengers. Returns 0 if no passengers."""
        if self.passengers == 0:
            return 0.0
        return round(self.??? / self.???, 2)

    # ── Display ─────────────────────────────────────────────
    def __str__(self):
        pk = "PEAK" if self.is_peak else "off-peak"
        return (f"Trip | Route {self.route.route_id} | "
                f"{self.hour:02d}:00 | "
                f"{self.passengers}/{self.route.capacity} pax | "
                f"₹{self.fare_collected} | {pk}")

    def __repr__(self):
        return (f"Trip(route={self.route.route_id!r}, "
                f"hour={self.hour}, passengers={self.passengers})")

### --- Test your Trip class ---


In [ ]:
r7  = Route("7",  "Harbour Express", 20.0, ["Central","Fort","Harbour","Navy Nagar","Colaba"], True,  60, "BEST")
r12 = Route("12", "Airport Shuttle", 35.0, ["Majestic","MG Road","Hebbal","Airport T1","Airport T2"], True, 45, "KSRTC")
r31 = Route("31", "City Loop",       12.0, ["KR Market","Shivajinagar","MG Road","Richmond","Jayanagar"], False, 80, "BMTC")

# Valid trips
t1 = Trip(r7,  8, 60)     # route 7, 8am, full bus
t2 = Trip(r12, 7, 40)     # route 12, 7am peak
t3 = Trip(r31, 14, 35)    # route 31, 2pm off-peak

print("=== Valid trips ===")
print(t1)
print(t2)
print(t3)

print("\n=== Computed properties ===")
print(f"t1 load        : {t1.load_pct()}%")
print(f"t1 overcrowded : {t1.is_overcrowded()}")
print(f"t2 rev/pax     : ₹{t2.revenue_per_passenger()}")

# Error handling — these should raise exceptions
print("\n=== Validation errors ===")
for bad_args, label in [
    ((r7,  25, 40), "hour=25 out of range"),
    ((r7,  8, -1),  "negative passengers"),
    ((r7,  8, 70),  "passengers > capacity"),
    (("7", 8, 40),  "route is a str not Route"),
]:
    try:
        t = Trip(*bad_args)
    except (ValueError, TypeError) as e:
        print(f"  ✅ Caught {type(e).__name__} for '{label}': {e}")

---
<div style="background:#fffaf0; border-left:5px solid #d97706; padding:20px 24px; border-radius:0 12px 12px 0; margin:24px 0 8px 0;">
  <h2 style="color:#92400e; margin:0 0 4px 0;">Part 4 · Inheritance</h2>
  <p style="color:#b45309; margin:0; font-size:14px;">⏱ ~35 minutes &nbsp;|&nbsp; Concepts: inheritance, super(), method overriding, isinstance(), polymorphism</p>
</div>

### 🧬 What is inheritance?

Inheritance lets one class **reuse all the code of another class**, then change or add only what is different. The child class gets everything the parent has — for free.


### 🔧 How inheritance works in code

The child class calls `super().__init__()` to set up all parent attributes, then adds its own. It can **override** methods to change behaviour while reusing parent logic via `super()`.


### 🔎 isinstance() — checking the type of an object

`isinstance(obj, ClassName)` returns `True` if the object is that class **or any child** of it. Polymorphism means you can treat all children the same — Python calls the right method automatically.


In [ ]:
#@title 🧬 Inheritance
from IPython.display import HTML, display
display(HTML("""
<div id="inh_w" style="max-width:620px;margin:16px auto;font-family:'Segoe UI',system-ui,sans-serif;background:linear-gradient(135deg,#1a1a2e 0%,#16213e 50%,#0f3460 100%);border-radius:14px;border:1px solid rgba(99,179,237,0.15);overflow:hidden;user-select:none;">
<div style="padding:14px 20px 6px;display:flex;justify-content:space-between;align-items:center;">
<span style="color:#90cdf4;font-size:13px;letter-spacing:1.5px;text-transform:uppercase;font-weight:600;">🧬 Inheritance</span>
<span id="inh_w_c" style="color:#4a5568;font-size:12px;font-family:monospace;"></span></div>
<div id="inh_w_s" style="padding:0 10px;min-height:300px;display:flex;align-items:center;justify-content:center;"></div>
<div style="padding:10px 20px 14px;display:flex;justify-content:center;align-items:center;gap:12px;">
<button id="inh_w_p" onclick="inh_w_n(-1)" style="background:rgba(99,179,237,0.1);border:1px solid rgba(99,179,237,0.3);color:#90cdf4;padding:8px 20px;border-radius:8px;cursor:pointer;font-size:14px;font-weight:600;" onmouseover="this.style.background='rgba(99,179,237,0.25)'" onmouseout="this.style.background='rgba(99,179,237,0.1)'">◀ Prev</button>
<div id="inh_w_d" style="display:flex;gap:6px;align-items:center;"></div>
<button id="inh_w_x" onclick="inh_w_n(1)" style="background:rgba(99,179,237,0.2);border:1px solid rgba(99,179,237,0.4);color:#fff;padding:8px 20px;border-radius:8px;cursor:pointer;font-size:14px;font-weight:600;" onmouseover="this.style.background='rgba(99,179,237,0.4)'" onmouseout="this.style.background='rgba(99,179,237,0.2)'">Next ▶</button></div></div>
<script>(function(){const id="inh_w";let step=0;const steps=[`<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 600 295" width="600" height="295" style="font-family:monospace;"><text x="300" y="20" fill="#b794f4" font-size="18" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">Inheritance tree</text><rect x="180" y="45" width="240" height="75" rx="10" fill="rgba(183,148,244,0.1)" stroke="rgb(183,148,244)" stroke-width="2"/><text x="300" y="68" fill="rgb(183,148,244)" font-size="18" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">Route</text><text x="300" y="95" fill="#a0aec0" font-size="10" text-anchor="middle" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">(parent / base class)</text><line x1="230" y1="120" x2="150" y2="145" stroke="#4a5568" stroke-width="2"/><line x1="370" y1="120" x2="450" y2="145" stroke="#4a5568" stroke-width="2"/><rect x="60" y="148" width="180" height="75" rx="10" fill="rgba(99,179,237,0.1)" stroke="rgb(99,179,237)" stroke-width="2"/><text x="150" y="170" fill="rgb(99,179,237)" font-size="15" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">ExpressBus</text><text x="150" y="195" fill="#90cdf4" font-size="10" text-anchor="middle" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">+ express_surcharge</text><rect x="360" y="148" width="180" height="75" rx="10" fill="rgba(154,230,180,0.1)" stroke="rgb(154,230,180)" stroke-width="2"/><text x="450" y="170" fill="rgb(154,230,180)" font-size="15" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">NightBus</text><text x="450" y="195" fill="#9ae6b4" font-size="10" text-anchor="middle" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">+ night_surcharge</text><text x="300" y="250" fill="#fbd38d" font-size="12" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">Children inherit everything from Route</text><text x="300" y="270" fill="#718096" font-size="11" text-anchor="middle" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">and can override or add new behaviour</text></svg>`,`<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 600 260" width="600" height="260" style="font-family:monospace;"><text x="300" y="18" fill="#63b3ed" font-size="16" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">How inheritance works in code</text><rect x="30" y="38" width="540" height="35" rx="6" fill="rgba(99,179,237,0.08)" stroke="rgba(99,179,237,0.2)" stroke-width="2"/><text x="50" y="58" fill="#e2e8f0" font-size="14" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">class ExpressBus(Route):</text><text x="320" y="58" fill="rgb(99,179,237)" font-size="11" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">← "inherits from Route"</text><rect x="30" y="85" width="540" height="50" rx="6" fill="rgba(154,230,180,0.06)" stroke="rgba(154,230,180,0.15)" stroke-width="2"/><text x="50" y="102" fill="#e2e8f0" font-size="12" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">def __init__(self, ..., express_surcharge=5.0):</text><text x="50" y="125" fill="#9ae6b4" font-size="13" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">    super().__init__(...)</text><text x="340" y="125" fill="rgb(154,230,180)" font-size="11" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">← calls parent's __init__</text><rect x="30" y="148" width="540" height="50" rx="6" fill="rgba(246,173,85,0.06)" stroke="rgba(246,173,85,0.15)" stroke-width="2"/><text x="50" y="165" fill="#e2e8f0" font-size="12" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">def get_fare(self, stops, is_peak):</text><text x="50" y="188" fill="#f6ad55" font-size="12" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">    base = super().get_fare(stops, is_peak)</text><text x="420" y="188" fill="#f6ad55" font-size="11" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">← reuse parent logic!</text><rect x="30" y="210" width="540" height="30" rx="6" fill="rgba(183,148,244,0.06)" stroke="rgba(183,148,244,0.15)" stroke-width="2"/><text x="50" y="227" fill="#b794f4" font-size="12" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">    return base + self.express_surcharge</text><text x="420" y="227" fill="#b794f4" font-size="11" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">← add own extra</text></svg>`,`<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 600 295" width="600" height="295" style="font-family:monospace;"><text x="300" y="18" fill="#76e4f7" font-size="16" text-anchor="middle" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">isinstance() and polymorphism</text><rect x="30" y="48" width="540" height="32" rx="4" fill="rgba(154,230,180,0.05)" stroke="rgba(154,230,180,0.12)" stroke-width="1"/><text x="45" y="66" fill="#a0aec0" font-size="11" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">r = Route(...)</text><text x="230" y="66" fill="#e2e8f0" font-size="12" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">isinstance(r, Route)</text><text x="465" y="66" fill="rgb(154,230,180)" font-size="13" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">True</text><rect x="30" y="88" width="540" height="32" rx="4" fill="rgba(99,179,237,0.05)" stroke="rgba(99,179,237,0.12)" stroke-width="1"/><text x="45" y="106" fill="#a0aec0" font-size="11" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">eb = ExpressBus(...)</text><text x="230" y="106" fill="#e2e8f0" font-size="12" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">isinstance(eb, ExpressBus)</text><text x="465" y="106" fill="rgb(154,230,180)" font-size="13" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">True</text><rect x="30" y="128" width="540" height="32" rx="4" fill="rgba(183,148,244,0.05)" stroke="rgba(183,148,244,0.12)" stroke-width="1"/><text x="230" y="146" fill="#e2e8f0" font-size="12" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">isinstance(eb, Route)</text><text x="465" y="146" fill="rgb(154,230,180)" font-size="13" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">True</text><text x="510" y="146" fill="#718096" font-size="9" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">also a Route!</text><rect x="30" y="168" width="540" height="32" rx="4" fill="rgba(252,129,129,0.05)" stroke="rgba(252,129,129,0.12)" stroke-width="1"/><text x="45" y="186" fill="#a0aec0" font-size="11" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">nb = NightBus(...)</text><text x="230" y="186" fill="#e2e8f0" font-size="12" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">isinstance(nb, ExpressBus)</text><text x="465" y="186" fill="rgb(252,129,129)" font-size="13" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">False</text><text x="510" y="186" fill="#718096" font-size="9" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">different child</text><rect x="30" y="218" width="540" height="55" rx="8" fill="rgba(251,211,141,0.06)" stroke="rgba(251,211,141,0.2)" stroke-width="2"/><text x="50" y="237" fill="#fbd38d" font-size="13" text-anchor="start" font-weight="bold" font-family="sans-serif" dominant-baseline="middle">Polymorphism:</text><text x="175" y="237" fill="#e2e8f0" font-size="11" text-anchor="start" font-weight="normal" font-family="monospace" dominant-baseline="middle">for route in [Route(...), ExpressBus(...), NightBus(...)]:</text><text x="60" y="260" fill="#fbd38d" font-size="12" text-anchor="start" font-weight="bold" font-family="monospace" dominant-baseline="middle">route.get_fare(6)</text><text x="230" y="260" fill="#a0aec0" font-size="10" text-anchor="start" font-weight="normal" font-family="sans-serif" dominant-baseline="middle">← calls the RIGHT method for each type automatically</text></svg>`];const sg=document.getElementById(id+"_s"),cn=document.getElementById(id+"_c"),dt=document.getElementById(id+"_d"),pb=document.getElementById(id+"_p"),nb=document.getElementById(id+"_x");for(let i=0;i<steps.length;i++){const d=document.createElement("div");d.style.cssText="width:8px;height:8px;border-radius:50%;background:#4a5568;transition:all .2s;cursor:pointer;";d.onclick=function(){step=i;r();};dt.appendChild(d);}function r(){sg.innerHTML=steps[step];cn.textContent="Step "+(step+1)+" of "+steps.length;pb.style.opacity=step===0?"0.3":"1";pb.style.pointerEvents=step===0?"none":"auto";nb.style.opacity=step===steps.length-1?"0.3":"1";nb.style.pointerEvents=step===steps.length-1?"none":"auto";for(let i=0;i<dt.children.length;i++){dt.children[i].style.background=i===step?"#63b3ed":"#4a5568";dt.children[i].style.transform=i===step?"scale(1.4)":"scale(1)";}}window[id+"_n"]=function(dir){step=Math.max(0,Math.min(steps.length-1,step+dir));r();};document.addEventListener("keydown",function(e){const rc=document.getElementById(id).getBoundingClientRect();if(rc.top>window.innerHeight||rc.bottom<0)return;if(e.key==="ArrowRight"||e.key===" "){e.preventDefault();window[id+"_n"](1);}if(e.key==="ArrowLeft"){e.preventDefault();window[id+"_n"](-1);}});r();})();</script>"""))

🧬 Inheritance 
 
 
 
 ◀ Prev 
 
 Next ▶

---
<div style="background:#1a1a2e; border-left:5px solid #f6ad55; padding:14px 18px; border-radius:0 10px 10px 0; margin:16px 0;">
  <strong style="color:#f6ad55;">🔴 INSTRUCTOR DEMO</strong> &nbsp; <span style="color:#a0aec0; font-size:14px;">Run each cell live while learners watch</span>
</div>


### DEMO — Inheritance — ExpressBus extends Route


In [ ]:
# INSTRUCTOR DEMO — Inheritance: ExpressBus extends Route

class ExpressBus(Route):
    """
    An express variant of Route with an additional flat surcharge.
    Always AC. Inherits all Route behaviour and overrides get_fare().
    """

    def __init__(self, route_id, name, base_fare, stops, capacity, operator,
                 express_surcharge=8.0):
        # super().__init__ sets up ALL Route attributes
        super().__init__(
            route_id  = route_id,
            name      = name,
            base_fare = base_fare,
            stops     = stops,
            is_ac     = True,           # ExpressBus is always AC
            capacity  = capacity,
            operator  = operator,
        )
        # Add the new attribute
        self.express_surcharge = express_surcharge

    def get_fare(self, stops_travelled, is_peak=False):
        """ExpressBus: parent fare + express surcharge."""
        base = super().get_fare(stops_travelled, is_peak)  # reuse Route logic
        return round(base + self.express_surcharge, 2)

    def __str__(self):
        return f"[EXPRESS] {super().__str__()} +₹{self.express_surcharge} surcharge"

### --- Create and compare ---


In [ ]:
standard = Route("12", "Airport Shuttle", 35.0,
                 ["Majestic","MG Road","Hebbal","Airport T1","Airport T2"],
                 True, 45, "KSRTC")

express  = ExpressBus("12X", "Airport Express", 35.0,
                      ["Majestic","Hebbal","Airport T1","Airport T2"],
                      40, "KSRTC", express_surcharge=12.0)

print("=== Standard vs Express ===")
print(f"Standard : {standard}")
print(f"Express  : {express}")

print("\n=== Fare comparison — 4 stops, peak ===")
print(f"Standard fare : ₹{standard.get_fare(4, is_peak=True)}")
print(f"Express fare  : ₹{express.get_fare(4, is_peak=True)}")

print("\n=== isinstance() checks ===")
print(f"express is Route       : {isinstance(express, Route)}")
print(f"express is ExpressBus  : {isinstance(express, ExpressBus)}")
print(f"standard is ExpressBus : {isinstance(standard, ExpressBus)}")

print("\n=== Polymorphism — mixed list, one loop ===")
fleet = [standard, express]
for bus in fleet:
    fare = bus.get_fare(4, is_peak=True)
    print(f"  {bus.route_id:<5} {bus.name:<22} peak fare: ₹{fare}")
    # Python automatically called the RIGHT get_fare for each object type

---
<div style="background:#1a3a1a; border-left:5px solid #68d391; padding:14px 18px; border-radius:0 10px 10px 0; margin:16px 0;">
  <strong style="color:#68d391;">🟢 YOUR TURN</strong> &nbsp; <span style="color:#a0aec0; font-size:14px;">Fill in every ??? — attempt before using AI</span>
</div>


### EXERCISE 3.4 — NightBus + mixed fleet (Mandatory)


In [ ]:
# YOUR TURN — Exercise 3.4: NightBus + mixed fleet
# Fill every ??? — the ExpressBus demo above is your pattern

### --- TASK 1: Build NightBus ---


In [ ]:
class NightBus(Route):
    """
    A late-night bus variant.
    Adds a flat night_surcharge to the base Route fare.
    NightBus is NOT necessarily AC — is_ac is passed in.
    is_peak is always False for NightBus (night hours only).
    """

    def __init__(self, route_id, name, base_fare, stops, is_ac,
                 capacity, operator, night_surcharge=5.0):
        # Call the parent constructor
        ???.__init__(
            route_id  = ???,
            name      = ???,
            base_fare = ???,
            stops     = ???,
            is_ac     = ???,
            capacity  = ???,
            operator  = ???,
        )
        self.night_surcharge = ???

    def get_fare(self, stops_travelled, is_peak=False):
        """
        NightBus fare = base Route fare (always off-peak) + night_surcharge.
        Ignore is_peak — night buses never apply the peak surcharge.
        """
        base = ???.get_fare(stops_travelled, is_peak=False)  # always off-peak
        return round(base + self.???, 2)

    def __str__(self):
        return f"[NIGHT] {???.__str__(self)} +₹{self.night_surcharge} night surcharge"

### --- TASK 2: Build a mixed fleet ---


In [ ]:
# Create 2 standard Routes, 1 ExpressBus, 1 NightBus
fleet = [
    Route("7", "Harbour Express", 20.0,
          ["Central Bus Stand","Fort","Harbour","Navy Nagar","Colaba"],
          True, 60, "BEST"),

    Route("31", "City Loop", 12.0,
          ["KR Market","Shivajinagar","MG Road","Richmond","Jayanagar"],
          False, 80, "BMTC"),

    ExpressBus("12X", "Airport Express", 35.0,
               ["Majestic","Hebbal","Airport T1","Airport T2"],
               40, "KSRTC", express_surcharge=12.0),

    NightBus("N7", "Night Harbour", 20.0,
             ["Central Bus Stand","Fort","Harbour","Colaba"],
             False, 50, "BEST", night_surcharge=6.0),
]

### --- TASK 3: Print a fleet summary table ---


In [ ]:
# Use isinstance() to label the bus type
print("=== CitySmart Fleet Summary ===")
print(f"  {'Route':<8} {'Name':<22} {'Type':<12} {'4-stop peak fare'}")
print(f"  {'-'*60}")

for bus in fleet:
    if isinstance(bus, ???):
        bus_type = "Express"
    elif isinstance(bus, ???):
        bus_type = "Night"
    else:
        bus_type = "Standard"

    fare = bus.get_fare(4, is_peak=True)   # same call on every bus type
    print(f"  {bus.route_id:<8} {bus.name:<22} {bus_type:<12} ₹{fare}")

### --- TASK 4: Polymorphism test ---


In [ ]:
# All buses in fleet respond to the same method calls
print("\n=== Full fleet descriptions ===")
for bus in fleet:
    print(f"  {bus}")      # each __str__ gives different output, same call

### --- TASK 5: isinstance() checks ---


In [ ]:
night = fleet[-1]   # NightBus
print(f"\n=== isinstance checks on NightBus ===")
print(f"  isinstance(night, Route)     : {isinstance(night, Route)}")
print(f"  isinstance(night, NightBus)  : {isinstance(night, NightBus)}")
print(f"  isinstance(night, ExpressBus): {isinstance(night, ExpressBus)}")

---
<div style="background:#2d1b00; border-left:5px solid #f6ad55; padding:20px 24px; border-radius:0 12px 12px 0; margin:24px 0 8px 0;">
  <h2 style="color:#fbd38d; margin:0 0 4px 0;">Advanced Exercises — Hours 3 & 4</h2>
  <p style="color:#f6ad55; margin:0; font-size:14px;">Combine files, error handling, and OOP into a complete, working system. Use AI as a reviewer and stress-tester, not a writer.</p>
</div>

### AI strategy for the advanced exercises

| What you need | Good AI prompt | Bad AI prompt |
|---|---|---|
| Understand a design decision | *"Should TransportNetwork store routes as a list or a dict? Give me the tradeoffs without writing any code."* | *"Write TransportNetwork for me"* |
| Generate stress-test data | *"Give me 10 edge-case inputs that could crash my load_trips_resilient() function. List them as bullet points."* | (testing only the happy path) |
| Code review | *"Review my `__init__` method. Don't rewrite it — list max 3 improvements in priority order."* | (skipping review) |
| Debug | *"My save() method writes the file but load() gives wrong types. Describe 3 possible causes — don't fix the code."* | *"Fix my save/load"* |

---
<div style="background:#1a3a1a; border-left:5px solid #68d391; padding:14px 18px; border-radius:0 8px 8px 0; margin:16px 0 4px 0;">
  <span style="color:#68d391; font-weight:700; font-size:15px;">🟢 Advanced Exercise A — The TransportNetwork class</span>
  <span style="color:#9ae6b4; font-size:13px; margin-left:12px;">25 minutes · intermediate → advanced</span>
</div>

**What you're building:** A `TransportNetwork` class that owns all Route objects and the trip log. It wraps the functions from Lessons 1–3 into a clean, object-oriented API. This is the central object of the CitySmart system.

> 💬 **AI Prompt — design before code:**
> *"I'm designing a Python class called TransportNetwork. It should: hold a dict of Route objects, hold a list of Trip objects, support saving/loading trips to CSV, and provide analytics methods (busiest route, hourly load, revenue breakdown). Ask me 3 clarifying questions about my design before suggesting anything."*
>
> Answer the questions. Build it yourself.

### ADVANCED EXERCISE A — TransportNetwork class (Optional)


In [ ]:
# ============================================================
# ADVANCED EXERCISE A — TransportNetwork class
# This is the central object of the entire CitySmart system
# ============================================================

class TransportNetwork:
    """
    The CitySmart transport network.
    Owns all Route objects and the Trip log.
    Provides analytics, file I/O, and search.
    """

    def __init__(self, name="CitySmart Network"):
        self.name   = name
        self.routes = {}    # {route_id: Route or subclass}
        self.trips  = []    # [Trip, Trip, ...]

    # ── Route management ────────────────────────────────────
    def add_route(self, route):
        """
        Add a Route (or subclass) to the network.
        Raises TypeError if route is not a Route instance.
        """
        if not isinstance(route, Route):
            raise TypeError(f"Expected Route, got {type(route).__name__}")
        self.routes[route.route_id] = route

    def get_route(self, route_id):
        """
        Return the Route for route_id.
        Raises RouteNotFoundError if not found.
        """
        if route_id not in self.routes:
            raise RouteNotFoundError(f"Route '{route_id}' not in network")
        return self.routes[route_id]

    # ── Trip logging ─────────────────────────────────────────
    def log_trip(self, route_id, hour, passengers):
        """
        Create and log a Trip. Uses Trip class validation.
        Returns the Trip object.
        """
        route = self.get_route(???)              # may raise RouteNotFoundError
        trip  = Trip(???, hour, passengers)      # may raise ValueError
        self.trips.???(trip)                     # add to internal list
        return trip

    # ── File persistence ─────────────────────────────────────
    def save_trips(self, filename="citysmart_trips.csv"):
        """
        Save all trips to CSV. Returns number of trips written.
        Each row: route_id, hour, passengers, capacity,
                  fare_collected, is_peak, is_ac
        """
        fieldnames = ["route_id","hour","passengers","capacity",
                      "fare_collected","is_peak","is_ac"]
        with open(filename, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            for trip in self.trips:
                writer.writerow({
                    "route_id"      : trip.route.route_id,
                    "hour"          : ???,
                    "passengers"    : ???,
                    "capacity"      : trip.route.capacity,
                    "fare_collected": ???,
                    "is_peak"       : ???,
                    "is_ac"         : trip.route.is_ac,
                })
        return len(self.trips)

    def load_trips(self, filename="citysmart_trips.csv"):
        """
        Load trips from CSV into self.trips.
        Skips rows with bad data (logs a warning per row).
        Returns (loaded_count, error_count).
        """
        loaded = 0
        errors = 0
        try:
            with open(filename, "r") as f:
                reader = csv.DictReader(f)
                for i, row in enumerate(reader, start=2):
                    try:
                        route_id   = row["route_id"]
                        hour       = int(row["hour"])
                        passengers = int(row["passengers"])
                        route = self.get_route(???)   # uses our validated getter
                        trip  = Trip(???, ???, ???)
                        self.trips.append(???)
                        loaded += 1
                    except (ValueError, KeyError, RouteNotFoundError,
                            CitySrnartError) as e:
                        print(f"  ⚠ Row {i} skipped: {e}")
                        errors += 1
        except FileNotFoundError:
            print(f"  ❌ File not found: {filename}")
        return loaded, errors

    # ── Analytics ────────────────────────────────────────────
    def busiest_route(self, metric="revenue"):
        """
        Return the (route_id, value) of the top route by metric.
        metric: "revenue" | "passengers" | "trips"
        """
        if not self.trips:
            return None
        totals = {}
        for trip in self.trips:
            rid = trip.route.route_id
            if metric == "revenue":
                val = trip.???
            elif metric == "passengers":
                val = trip.???
            else:
                val = 1
            totals[rid] = totals.get(rid, 0) + val
        return max(totals.items(), key=lambda x: x[1])

    def capacity_alerts(self, threshold=0.95):
        """Return all trips where load >= threshold."""
        return [t for t in self.trips if t.load_pct() / 100 >= threshold]

    def __str__(self):
        return (f"{self.name} | {len(self.routes)} routes | "
                f"{len(self.trips)} trips logged")


# ── BUILD AND TEST THE NETWORK ────────────────────────────────
network = TransportNetwork("CitySmart Bangalore")

# Add routes (mix of types)
for rid, info in NETWORK.items():
    network.add_route(Route(
        rid, info["name"], info["base_fare"], info["stops"],
        info["is_ac"], info["capacity"], info["operator"]
    ))
network.add_route(ExpressBus("12X","Airport Express",35.0,
                  ["Majestic","Hebbal","Airport T1","Airport T2"],
                  40,"KSRTC",express_surcharge=12.0))
network.add_route(NightBus("N7","Night Harbour",20.0,
                  ["Central Bus Stand","Fort","Harbour","Colaba"],
                  False,50,"BEST",night_surcharge=6.0))

# Log trips
for rid, hour, pax in raw_trips:
    try:
        network.log_trip(rid, hour, pax)
    except (RouteNotFoundError, ValueError):
        pass

print(network)
print(f"\nBusiest route (revenue)   : {network.busiest_route('revenue')}")
print(f"Busiest route (passengers): {network.busiest_route('passengers')}")

alerts = network.capacity_alerts()
print(f"\nCapacity alerts (≥95%): {len(alerts)}")
for t in alerts:
    print(f"  {t}")

# Save and reload
n = network.save_trips("network_trips.csv")
print(f"\n✅ Saved {n} trips")

network2 = TransportNetwork("CitySmart Reload Test")
for rid, info in NETWORK.items():
    network2.add_route(Route(rid,info["name"],info["base_fare"],
                             info["stops"],info["is_ac"],
                             info["capacity"],info["operator"]))

loaded, errs = network2.load_trips("network_trips.csv")
print(f"✅ Loaded {loaded} trips, {errs} errors")
print(f"✅ Revenue check: {network2.busiest_route('revenue')}")

---
<div style="background:#1a3a1a; border-left:5px solid #68d391; padding:14px 18px; border-radius:0 8px 8px 0; margin:16px 0 4px 0;">
  <span style="color:#68d391; font-weight:700; font-size:15px;">🟢 Advanced Exercise B — AI stress-test: break your own code</span>
  <span style="color:#9ae6b4; font-size:13px; margin-left:12px;">20 minutes · advanced</span>
</div>

**What you're doing:** Using AI as an adversary. Ask it to generate every edge case that could break your `load_trips()` method, then fix every one you find.

The discipline: **you fix — AI only finds.** Never paste an AI fix directly.

> 💬 **AI Prompt — run this first:**
> *"I have a Python method load_trips() that reads a CSV file of bus trip records and creates Trip objects. It handles FileNotFoundError, ValueError, and KeyError. Generate exactly 8 edge-case inputs that could still cause it to fail or produce wrong data. Format each as: (1) what the bad input is, (2) which specific line of code it would break, (3) what exception or wrong result it would produce. Don't fix any of them."*
>
> For each issue AI identifies: reproduce it yourself in the cell below, then fix it in your `TransportNetwork.load_trips()`.

### ADVANCED EXERCISE B — AI stress-test your loader (Optional)


In [ ]:
# ============================================================
# ADVANCED EXERCISE B — AI stress-test your load_trips()
# ============================================================

# ── STEP 1: Paste AI's 8 edge cases here as comments ─────────
# Work through each one:
#
# Edge case 1: ________________________________________________
#   Breaks at: ________________________________________________
#   Because  : ________________________________________________
#   Fixed    : ________________________________________________
#
# Edge case 2: ________________________________________________
#   Breaks at: ________________________________________________
#   Because  : ________________________________________________
#   Fixed    : ________________________________________________
#
# ... (continue for all 8)


# ── STEP 2: Reproduce each edge case as a CSV and test ───────
# For each edge case AI found, write a minimal CSV that triggers it.
# Use load_trips_resilient() or network.load_trips() on each file.

# Example template — replace with each of your 8 edge cases:
edge_case_1_csv = """route_id,hour,passengers,capacity,fare_collected,is_peak,is_ac
7,8,,60,1380.0,True,True
"""

with open("edge_case_1.csv", "w") as f:
    f.write(edge_case_1_csv)

net_test = TransportNetwork("Test")
for rid, info in NETWORK.items():
    net_test.add_route(Route(rid,info["name"],info["base_fare"],
                             info["stops"],info["is_ac"],
                             info["capacity"],info["operator"]))

loaded, errs = net_test.load_trips("edge_case_1.csv")
print(f"Edge case 1: loaded={loaded}, errors={errs}")

# Repeat for edge cases 2–8:
# ...


# ── STEP 3: Document what you fixed ──────────────────────────
# Fill this in after fixing:
print("\n=== Fixes applied to TransportNetwork.load_trips() ===")
fixes = [
    "Fix 1: ...",   # describe each fix you made
    "Fix 2: ...",
    # ...
]
for fix in fixes:
    print(f"  ✅ {fix}")

---
<div style="background:#1a3a1a; border-left:5px solid #68d391; padding:14px 18px; border-radius:0 8px 8px 0; margin:16px 0 4px 0;">
  <span style="color:#68d391; font-weight:700; font-size:15px;">🟢 Advanced Exercise C — The complete OOP system (capstone)</span>
  <span style="color:#9ae6b4; font-size:13px; margin-left:12px;">30 minutes · advanced · Week 3 deliverable</span>
</div>

**What you're building:** Wire everything together into a single runnable demonstration. Build the full system: mixed fleet, log a day of trips, save to CSV, reload in a fresh network object, run analytics, print the dashboard — all using the OOP layer.

This is your **Week 3 deliverable** — a self-contained Python program that runs from top to bottom without any globals.

> 💬 **AI Prompt — only if you get stuck on the dashboard integration:**
> *"My print_dashboard() function from Lesson 2 expects a list of plain dicts. My new Trip objects are not dicts. What are my options for making them compatible — without rewriting print_dashboard()? Give me three approaches and the tradeoff of each."*

### ADVANCED EXERCISE C — The complete OOP system (Optional)


In [ ]:
# ============================================================
# ADVANCED EXERCISE C — The complete OOP system
# Run top to bottom. No globals. Everything through the network object.
# ============================================================

# ── STEP 1: Build the full mixed-fleet network ───────────────
def build_citysmart_network():
    """Build and return a fully configured TransportNetwork."""
    net = TransportNetwork("CitySmart Bangalore — Full System")

    # Standard routes
    net.add_route(Route("7",  "Harbour Express",  20.0, ["Central Bus Stand","Fort","Harbour","Navy Nagar","Colaba"],            True,  60, "BEST"))
    net.add_route(Route("12", "Airport Shuttle",  35.0, ["Majestic","MG Road","Hebbal","Airport T1","Airport T2"],              True,  45, "KSRTC"))
    net.add_route(Route("31", "City Loop",        12.0, ["KR Market","Shivajinagar","MG Road","Richmond","Jayanagar"],          False, 80, "BMTC"))
    net.add_route(Route("42", "North Connector",  18.0, ["Yeshwanthpur","Peenya","Tumkur Road","Nelamangala","Dobbaspet"],      False, 80, "BMTC"))
    net.add_route(Route("55", "South Express",    15.0, ["Majestic","Jayanagar","JP Nagar","Bannerghatta Rd","Electronic City"],True,  60, "BMTC"))
    net.add_route(Route("88", "Tech Corridor",    22.0, ["Silk Board","HSR Layout","Koramangala","Indiranagar","MG Road"],      True,  50, "KSRTC"))

    # Express routes
    net.add_route(ExpressBus("12X", "Airport Express", 35.0,
                  ["Majestic","Hebbal","Airport T1","Airport T2"],
                  40, "KSRTC", express_surcharge=12.0))

    # Night routes
    net.add_route(NightBus("N7", "Night Harbour", 20.0,
                  ["Central Bus Stand","Fort","Harbour","Colaba"],
                  False, 50, "BEST", night_surcharge=6.0))
    net.add_route(NightBus("N31", "Night Loop", 12.0,
                  ["KR Market","MG Road","Jayanagar"],
                  False, 60, "BMTC", night_surcharge=4.0))

    return net


# ── STEP 2: Log a full day including express and night buses ──
def simulate_day(net):
    """Log all trips — including express and night routes."""
    day_data = [
        ("7",6,12), ("7",7,54), ("7",8,60), ("7",9,58),
        ("7",12,32),("7",17,55),("7",18,59),("7",21,18),
        ("12",5,8), ("12",7,40),("12",8,44),("12",9,43),
        ("12",13,20),("12",17,42),("12",18,45),("12",20,30),
        ("31",6,45),("31",7,78),("31",8,80),("31",9,75),
        ("31",11,40),("31",14,35),("31",17,79),("31",19,72),
        ("42",6,30),("42",7,72),("42",8,78),("42",17,74),
        ("42",18,76),("42",21,28),
        ("55",7,55),("55",8,58),("55",9,52),("55",17,56),
        ("55",18,59),("55",20,35),
        ("88",7,48),("88",8,50),("88",9,49),("88",17,47),
        ("88",18,50),("88",19,44),
        # Express
        ("12X",6,30),("12X",7,38),("12X",8,40),("12X",9,39),
        ("12X",17,37),("12X",18,40),
        # Night
        ("N7",22,20),("N7",23,15),("N31",22,35),("N31",23,28),
    ]
    logged = 0
    for rid, hour, pax in day_data:
        try:
            net.log_trip(rid, hour, pax)
            logged += 1
        except (RouteNotFoundError, ValueError, CapacityExceededError) as e:
            print(f"  ⚠ Skipped trip {rid}/{hour}/{pax}: {e}")
    return logged


# ── STEP 3: Print the OOP analytics dashboard ─────────────────
def print_oop_dashboard(net):
    """Print a dashboard using net.trips (list of Trip objects)."""
    W = 60
    def bar(v, mx, w=26): return "█"*int(v/mx*w if mx else 0) + "░"*(w-int(v/mx*w if mx else 0))
    def section(h): print(f"\n  {h}"); print(f"  {'-'*(W-2)}")

    total_rev  = sum(t.fare_collected for t in net.trips)
    total_pax  = sum(t.passengers     for t in net.trips)
    peak_rev   = sum(t.fare_collected for t in net.trips if t.is_peak)
    peak_share = round(peak_rev / total_rev * 100, 1) if total_rev else 0

    print("=" * W)
    print(f"  🚌  {net.name}")
    print(f"  {'Fleet size:':<28} {len(net.routes)} routes")
    print(f"  {'Trips logged:':<28} {len(net.trips)}")
    print(f"  {'Express routes:':<28} {sum(1 for r in net.routes.values() if isinstance(r, ExpressBus))}")
    print(f"  {'Night routes:':<28} {sum(1 for r in net.routes.values() if isinstance(r, NightBus))}")
    print("=" * W)

    section("📊  Fleet Summary")
    print(f"  {'Total revenue':.<30} ₹{total_rev:>10,.2f}")
    print(f"  {'Total passengers':.<30} {total_pax:>11,}")
    print(f"  {'Peak hour share':.<30} {peak_share:>10.1f}%")

    section("💰  Top 3 Routes by Revenue")
    rev_by_route = {}
    for t in net.trips:
        rid = t.route.route_id
        rev_by_route[rid] = rev_by_route.get(rid, 0) + t.fare_collected
    top3 = sorted(rev_by_route.items(), key=lambda x: -x[1])[:3]
    max_rev = top3[0][1] if top3 else 1
    for rid, rev in top3:
        print(f"  Route {rid:<5} {bar(rev,max_rev)}  ₹{rev:,.0f} ({net.routes[rid].name})")

    section("⚠  Capacity Alerts (≥ 95%)")
    alerts = net.capacity_alerts(0.95)
    if alerts:
        for t in sorted(alerts, key=lambda x: -x.load_pct()):
            pk = "PEAK" if t.is_peak else "off-peak"
            print(f"  Route {t.route.route_id} at {t.hour:02d}:00 — {t.load_pct():.0f}% [{pk}]")
    else:
        print("  No alerts ✓")

    section("🚀  Express vs Night Revenue")
    expr_rev  = sum(t.fare_collected for t in net.trips if isinstance(t.route, ExpressBus))
    night_rev = sum(t.fare_collected for t in net.trips if isinstance(t.route, NightBus))
    std_rev   = total_rev - expr_rev - night_rev
    print(f"  Standard routes : ₹{std_rev:>10,.2f}")
    print(f"  Express routes  : ₹{expr_rev:>10,.2f}")
    print(f"  Night routes    : ₹{night_rev:>10,.2f}")

    print(f"\n  {'-'*(W-2)}")
    print(f"  System check: {len(net.trips)} trips | ₹{total_rev:,.2f} total")
    print("=" * W)


# ── RUN THE COMPLETE SYSTEM ───────────────────────────────────
net = build_citysmart_network()
n   = simulate_day(net)
print(f"✅ Logged {n} trips\n")

print_oop_dashboard(net)

# Save the OOP system state
saved = net.save_trips("citysmart_oop_trips.csv")
print(f"\n✅ Saved {saved} trips to citysmart_oop_trips.csv")

# Reload into a fresh network and verify
net2 = build_citysmart_network()
loaded, errs = net2.load_trips("citysmart_oop_trips.csv")
print(f"✅ Reloaded: {loaded} trips, {errs} errors")

rev_original = sum(t.fare_collected for t in net.trips)
rev_reloaded = sum(t.fare_collected for t in net2.trips)
print(f"✅ Revenue match: {abs(rev_original - rev_reloaded) < 0.01}")

---
<div style="background:#f0fff4; border:1.5px solid #38a169; border-radius:12px; padding:24px 28px; margin:24px 0;">

## 🏁 Week 3 Deliverable — What you've built

Run Exercise C top to bottom and confirm:

- ✅ `build_citysmart_network()` creates a mixed fleet (standard, express, night buses)
- ✅ `simulate_day()` logs all trips through `TransportNetwork.log_trip()` with validation
- ✅ `save_trips()` writes all trips to CSV
- ✅ `load_trips()` reloads them into a fresh `TransportNetwork` — types all correct
- ✅ Revenue totals match between original and reloaded networks
- ✅ `print_oop_dashboard()` runs analytics entirely on `Trip` objects — no raw dicts
- ✅ The system never crashes — bad data is caught and logged, not fatal

**Save your notebook** — Lesson 4 loads `citysmart_oop_trips.csv` into NumPy and Pandas for a full EDA.

</div>

---

## 🔑 Key concepts recap

| Concept | What it is | Where you used it |
|---|---|---|
| `open(f, mode)` | Open a file for reading or writing | save_trips, load_trips |
| `with` statement | Auto-closes file even on crash | Every file operation |
| `csv.DictWriter` | Write dicts as CSV rows | save_trips() |
| `csv.DictReader` | Read CSV rows as dicts | load_trips() |
| Type casting from CSV | Every value is a string — cast manually | `int(row["hour"])` |
| `json.dump / load` | Save/restore nested structures | network_demo.json |
| `try / except` | Catch and handle exceptions | All file and validation code |
| `except E1, E2` | Catch multiple types at once | `(ValueError, KeyError)` |
| `finally` | Runs regardless of error | Cleanup code |
| `raise` | Deliberately signal a problem | `log_trip_strict()`, `__init__` |
| Custom exceptions | Named exception classes | `RouteNotFoundError`, `CapacityExceededError` |
| `class` | Blueprint for objects | `Route`, `Trip`, `TransportNetwork` |
| `__init__` | Constructor — runs on object creation | All classes |
| `self` | The specific object being operated on | Every method |
| Instance attribute | Data stored on a specific object | `self.base_fare`, `self.stops` |
| Instance method | Function that works on `self` | `get_fare()`, `serves_stop()` |
| `__str__` | Human-readable display string | `print(route)` |
| `__repr__` | Developer-readable representation | REPL, inside lists |
| Inheritance | Child class reuses parent code | `ExpressBus(Route)`, `NightBus(Route)` |
| `super()` | Call the parent class method | `super().__init__()`, `super().get_fare()` |
| Method overriding | Child redefines parent method | `get_fare()` in ExpressBus and NightBus |
| `isinstance()` | Check object type at runtime | Bus type detection in dashboard |
| Polymorphism | One call, different behaviour per type | `bus.get_fare()` on mixed fleet |

---

## 📚 Before Lesson 4

1. Add one more route type — `LuxuryBus` that inherits from `ExpressBus` and adds a `luxury_amenity_fee`. Override `__str__` to show the amenity fee.
2. Add a `to_dict()` method to `Trip` that returns a plain dict with all key fields — this will be used in Lesson 4 to convert Trip objects into Pandas rows.
3. Make `TransportNetwork.capacity_alerts()` accept a `route_type` parameter that filters by `Route`, `ExpressBus`, or `NightBus` — use `isinstance()`.
4. Ask AI: *"I have a CSV file of 10,000 bus trip records. I want to analyse it with Pandas. What are the first 5 things I should do after loading it with pd.read_csv()? Give me the pandas method for each step — no explanation, just the method names and a one-line description."* Come ready to use those methods in Lesson 4.